# 전체 USPTO 특허 Feature 구축

GNN 학습을 위한 **전체 utility 특허 (1976~2016)** 대상 피처 추출.  
참조: `20260609 특허소송예측 데이터구축.ipynb`

| 항목 | 내용 |
|------|------|
| 대상 | PatentsView utility 비취하 특허, grant_date ≤ 2016-12-31 |
| 피처 | MCD, SCD, TD_A~H, ABS, CTS, PTS, CTC, PK, SK, PPS, CD, IC, IE, FIE, ET |
| 출력 | `./data_gnn/patent_features.parquet`, `./data_gnn/patent_company_map.parquet` |
| 데이터 | PatentsView TSV + PACER CSV: `../src/` |

**LI1~LI7 제외**: GNN이 citation+company 그래프 구조에서 소송 전파 패턴을 직접 학습.  
**TCP 제외**: 소송 특허에만 원고 기반 값이 존재해 label leakage 위험.  
**CS 제외**: g_patent_abroad.tsv 미보유, PCT proxy 커버리지 낮아 제외.

---
**처리 순서**
1. PACER → 소송 특허 집합 (`is_litigated` 플래그용)
2. g_patent.tsv → utility 비취하 특허 (~2016)
3. g_assignee_disambiguated.tsv → Patent-Company 매핑, CD, IC
4. 개별 TSV → PK, SK, PPS, ET, IE, FIE, ABS, IPC(MCD/SCD/TD_A~H)
5. g_claims_*.tsv → CTS, PTS, CTC
6. 통합 & 저장

In [4]:
# [Cell 1]
import pandas as pd
import numpy as np
import re, os, gc
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

os.nice(10)
os.environ['OMP_NUM_THREADS']  = '2'
os.environ['MKL_NUM_THREADS']  = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', 500)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

SRC_DIR  = '../src'
DATA_DIR = './data_gnn'
os.makedirs(DATA_DIR, exist_ok=True)

CHUNKSIZE = 300_000

PATENT_FEAT_COLS = [
    'MCD', 'SCD',
    'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H',
    'ABS', 'CTS', 'CTC', 'PTS',
    'PK', 'SK', 'PPS',
    'CD', 'IC', 'IE', 'FIE', 'ET',
]

print('환경 설정 완료')
print(f'피처 수: {len(PATENT_FEAT_COLS)}개')
print(f'CHUNKSIZE: {CHUNKSIZE:,}')
print()
print('★ 무거운 순서: Cell7(PK) >> Cell10(IE/FIE) >> Cell12(IPC) >> Cell14(Claims)')

환경 설정 완료
피처 수: 22개
CHUNKSIZE: 300,000

★ 무거운 순서: Cell7(PK) >> Cell10(IE/FIE) >> Cell12(IPC) >> Cell14(Claims)


---
## Step 1. 소송 특허 집합 추출 (is_litigated 플래그용)

In [ ]:
# [Cell 2]
cases   = pd.read_csv(f'{SRC_DIR}/cases.csv',   low_memory=False)
patents = pd.read_csv(f'{SRC_DIR}/patents.csv',  low_memory=False)

inf_cases = cases[cases['case_type_1'] == 1.0]

lit_patents_df = patents[
    (patents['case_row_id'].isin(inf_cases['case_row_id'])) &
    (patents['patent_doc_type'].str.strip() == 'Patent')
].copy()

def normalize_patent(s):
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    m = re.match(r'^([A-Z]*)(\d+)$', s)
    if not m:
        return None
    return m.group(1) + m.group(2).lstrip('0')

lit_patents_df['patent_norm'] = lit_patents_df['patent'].astype(str).apply(normalize_patent)
litigated_set_int = {int(p) for p in lit_patents_df['patent_norm'].dropna() if p.isdigit()}

print(f'침해소송 케이스: {len(inf_cases):,}건')
print(f'소송 특허 (정수 변환): {len(litigated_set_int):,}건')

---
## Step 2. Company 정규화 함수

In [ ]:
# [Cell 3]
# deep_normalize: 20260625 NPE 정확매칭 노트북 기준
_SUFFIX = re.compile(
    r',?\s*(incorporated|corporation|company|limited|enterprises|'
    r'inc|corp|llc|ltd|co|plc|lp|llp|gmbh|ag|sa|sl|srl|bv|nv|ab|as|kk|'
    r'the|terminated.*|dissolved.*|successor.*)[,.]?\s*$',
    re.IGNORECASE
)
_PREFIX = re.compile(r'^(the)\s+', re.IGNORECASE)

def deep_normalize(name):
    if pd.isna(name):
        return None
    s = str(name).strip().replace('&', 'and')
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip().upper()
    for _ in range(6):
        new = _SUFFIX.sub('', s, count=1).strip()
        if new == s:
            break
        s = new
    return _PREFIX.sub('', s).strip() or None

print('deep_normalize 준비 완료')

---
## Step 3. 전체 Utility 특허 목록

In [ ]:
# [Cell 4]
print('g_patent.tsv 로딩...', flush=True)
pat_base = pd.read_csv(
    f'{SRC_DIR}/g_patent.tsv', sep='\t',
    usecols=['patent_id', 'patent_type', 'patent_date', 'num_claims', 'withdrawn'],
    low_memory=False
)
pat_base['patent_id'] = pd.to_numeric(pat_base['patent_id'], errors='coerce')
pat_base = pat_base.dropna(subset=['patent_id']).astype({'patent_id': int})

pat_util = pat_base[
    (pat_base['patent_type'] == 'utility') &
    (pat_base['withdrawn'] == 0) &
    (pat_base['patent_date'] <= '2016-12-31')   # 소송 데이터 관측 기간 기준
].copy()
pat_util['grant_date'] = pd.to_datetime(pat_util['patent_date'], errors='coerce')

all_pids = set(pat_util['patent_id'])

print(f'전체 PatentsView 특허:  {len(pat_base):,}건')
print(f'  utility + 비취하 + ~2016: {len(pat_util):,}건')
print(f'  소송 특허 포함:           {len(all_pids & litigated_set_int):,}건')

---
## Step 4. Patent-Company 매핑 + CD / IC 계산

In [ ]:
# [Cell 5]
print('g_assignee_disambiguated.tsv 로딩...', flush=True)
asgn_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_assignee_disambiguated.tsv', sep='\t',
    usecols=['patent_id', 'assignee_sequence', 'disambig_assignee_organization', 'location_id'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        asgn_chunks.append(chunk)

asgn_df = pd.concat(asgn_chunks, ignore_index=True)
asgn_df['org_norm'] = asgn_df['disambig_assignee_organization'].apply(deep_normalize)

# CD: 특허당 출원인 수
cd_df = (
    asgn_df.groupby('patent_id')['assignee_sequence'].count()
    .reset_index().rename(columns={'assignee_sequence': 'CD'})
)

# IC: 특허당 출원인 국가 수 (location_id → raw_country)
need_locs = set(asgn_df['location_id'].dropna().unique())
loc_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_location_not_disambiguated.tsv', sep='\t',
    usecols=['location_id', 'raw_country'],
    low_memory=False, chunksize=CHUNKSIZE
):
    c = chunk[chunk['location_id'].isin(need_locs)]
    if len(c) > 0:
        loc_chunks.append(c)
loc_df = pd.concat(loc_chunks).drop_duplicates('location_id')[['location_id', 'raw_country']]
ic_df = (
    asgn_df.merge(loc_df, on='location_id', how='left')
    .groupby('patent_id')['raw_country'].nunique()
    .reset_index().rename(columns={'raw_country': 'IC'})
)

# 특허당 대표 출원인 (assignee_sequence 최소 = 주 출원인)
patent_company = (
    asgn_df.sort_values('assignee_sequence')
    .dropna(subset=['org_norm'])
    .drop_duplicates('patent_id')
    [['patent_id', 'org_norm']]
)

print(f'Assignee 로딩 완료: {len(asgn_df):,}행')
print(f'CD: {len(cd_df):,}건 / IC: {len(ic_df):,}건')
print(f'Patent-Company 매핑: {len(patent_company):,}건')
del asgn_chunks; gc.collect()

In [ ]:
# [Cell 6]
import os

def load_ckpt(path, name):
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f'  ✓ {name} 로드 ({len(df):,}건) ← {path}')
        return df
    else:
        print(f'  ✗ {name} 체크포인트 없음 → 해당 셀 직접 실행 필요')
        return None

print('=== 체크포인트 현황 ===')
if 'pk_df' not in dir():
    pk_df = load_ckpt(f'{DATA_DIR}/ckpt_pk.parquet', 'PK')
if 'sk_df' not in dir():
    _tmp = load_ckpt(f'{DATA_DIR}/ckpt_sk_pps.parquet', 'SK+PPS')
    if _tmp is not None:
        sk_df  = _tmp[['patent_id','SK']].dropna(subset=['SK'])
        pps_df = _tmp[['patent_id','PPS']].dropna(subset=['PPS'])
if 'et_df' not in dir():
    et_df = load_ckpt(f'{DATA_DIR}/ckpt_et.parquet', 'ET')
if 'ie_df' not in dir():
    _tmp = load_ckpt(f'{DATA_DIR}/ckpt_ie_fie.parquet', 'IE+FIE')
    if _tmp is not None:
        ie_df  = _tmp[['patent_id','IE']].dropna(subset=['IE'])
        fie_df = _tmp[['patent_id','FIE']].dropna(subset=['FIE'])
if 'abs_df' not in dir():
    abs_df = load_ckpt(f'{DATA_DIR}/ckpt_abs.parquet', 'ABS')
if 'ipc_feat' not in dir():
    ipc_feat = load_ckpt(f'{DATA_DIR}/ckpt_ipc.parquet', 'IPC')
if 'claims_feat' not in dir():
    claims_feat = load_ckpt(f'{DATA_DIR}/ckpt_claims.parquet', 'Claims')

---
## Step 5. PatentsView 개별 피처 추출

In [ ]:
# [Cell 7]
# PK: 인용한 선행기술 수 (outgoing citations) — 대용량 chunked
# ★ 가장 오래 걸리는 단계 (수 시간 소요 가능)
print('PK 추출 중... (대용량 파일, 시간 소요)', flush=True)
pk_acc = {}
for i, chunk in enumerate(pd.read_csv(
    f'{SRC_DIR}/g_us_patent_citation.tsv', sep='\t',
    usecols=['patent_id', 'citation_sequence'],
    low_memory=False, chunksize=CHUNKSIZE
)):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    for pid, cnt in chunk.groupby('patent_id').size().items():
        pk_acc[int(pid)] = pk_acc.get(int(pid), 0) + int(cnt)
    if i % 20 == 0:
        print(f'  chunk {i:3d}: {len(pk_acc):,}건 누적', flush=True)

pk_df = pd.DataFrame(list(pk_acc.items()), columns=['patent_id', 'PK'])
pk_df.to_parquet(f'{DATA_DIR}/ckpt_pk.parquet', index=False)
print(f'PK 완료: {len(pk_df):,}건  →  ckpt_pk.parquet 저장')
del pk_acc; gc.collect()

In [ ]:
# [Cell 8]
# SK: 비특허 선행기술 인용 수
print('SK 추출...', flush=True)
sk_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_other_reference.tsv', sep='\t',
    usecols=['patent_id', 'other_reference_sequence'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        sk_chunks.append(chunk)
sk_df = (
    pd.concat(sk_chunks).groupby('patent_id').size()
    .reset_index().rename(columns={0: 'SK'})
)
print(f'SK: {len(sk_df):,}건')
del sk_chunks; gc.collect()

# PPS: 외국 우선권 국가 수
print('PPS 추출...', flush=True)
pps_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_foreign_priority.tsv', sep='\t',
    usecols=['patent_id', 'foreign_country_filed'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        pps_chunks.append(chunk)
pps_df = (
    pd.concat(pps_chunks).groupby('patent_id')['foreign_country_filed'].nunique()
    .reset_index().rename(columns={'foreign_country_filed': 'PPS'})
)
print(f'PPS: {len(pps_df):,}건')
del pps_chunks; gc.collect()

pd.merge(sk_df, pps_df, on='patent_id', how='outer').to_parquet(f'{DATA_DIR}/ckpt_sk_pps.parquet', index=False)
print('ckpt_sk_pps.parquet 저장')

In [ ]:
# [Cell 9]
# ET: 심사 기간 (grant_date - filing_date, 일)
print('ET 추출...', flush=True)
app_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_application.tsv', sep='\t',
    usecols=['patent_id', 'filing_date'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        app_chunks.append(chunk)

app_df = pd.concat(app_chunks).drop_duplicates('patent_id')
et_df = pat_util[['patent_id', 'grant_date']].merge(app_df, on='patent_id', how='inner')

filing_dt = pd.to_datetime(et_df['filing_date'], errors='coerce')
filing_dt = filing_dt.where(filing_dt >= '1970-01-01', other=pd.NaT)  # 소송 특허 최초 출원 1973년 기준

et_df['ET'] = (et_df['grant_date'] - filing_dt).dt.days
et_df = et_df[['patent_id', 'ET']].dropna()
et_df.to_parquet(f'{DATA_DIR}/ckpt_et.parquet', index=False)
print(f'ET: {len(et_df):,}건  →  ckpt_et.parquet 저장')
del app_chunks; gc.collect()

In [ ]:
# [Cell 10]
# IE: 발명자 수 / FIE: 발명자 국가 수
print('IE / FIE 추출...', flush=True)
inv_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_inventor_not_disambiguated.tsv', sep='\t',
    usecols=['patent_id', 'inventor_sequence', 'rawlocation_id'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        inv_chunks.append(chunk)

inv_df = pd.concat(inv_chunks, ignore_index=True)
ie_df = (
    inv_df.groupby('patent_id')['inventor_sequence'].count()
    .reset_index().rename(columns={'inventor_sequence': 'IE'})
)

need_locs_inv = set(inv_df['rawlocation_id'].dropna().unique())
loc2_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_location_not_disambiguated.tsv', sep='\t',
    usecols=['rawlocation_id', 'raw_country'],
    low_memory=False, chunksize=CHUNKSIZE
):
    c = chunk[chunk['rawlocation_id'].isin(need_locs_inv)]
    if len(c) > 0:
        loc2_chunks.append(c)
loc2_df = pd.concat(loc2_chunks).drop_duplicates('rawlocation_id')

fie_df = (
    inv_df.merge(loc2_df, on='rawlocation_id', how='left')
    .groupby('patent_id')['raw_country'].nunique()
    .reset_index().rename(columns={'raw_country': 'FIE'})
)
print(f'IE: {len(ie_df):,}건 / FIE: {len(fie_df):,}건')
del inv_chunks, inv_df; gc.collect()

pd.merge(ie_df, fie_df, on='patent_id', how='outer').to_parquet(f'{DATA_DIR}/ckpt_ie_fie.parquet', index=False)
print('ckpt_ie_fie.parquet 저장')

In [ ]:
# [Cell 11]
# ABS: 초록 단어 수
print('ABS 추출...', flush=True)
abs_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_patent_abstract.tsv', sep='\t',
    usecols=['patent_id', 'patent_abstract'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        abs_chunks.append(chunk)

abs_raw = pd.concat(abs_chunks)
abs_raw['ABS'] = abs_raw['patent_abstract'].fillna('').apply(
    lambda x: len(str(x).split())
)
abs_df = abs_raw[['patent_id', 'ABS']].drop_duplicates('patent_id')
abs_df.to_parquet(f'{DATA_DIR}/ckpt_abs.parquet', index=False)
print(f'ABS: {len(abs_df):,}건  →  ckpt_abs.parquet 저장')
del abs_chunks, abs_raw; gc.collect()

In [ ]:
# [Cell 12]
# IPC: MCD(주분류 수), SCD(부분류 수), TD_A~H(섹션 유무)
print('IPC 추출...', flush=True)
ipc_chunks = []
for chunk in pd.read_csv(
    f'{SRC_DIR}/g_ipc_at_issue.tsv', sep='\t',
    usecols=['patent_id', 'section', 'ipc_class', 'subclass'],
    low_memory=False, chunksize=CHUNKSIZE
):
    chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
    chunk = chunk[chunk['patent_id'].isin(all_pids)]
    if len(chunk) > 0:
        ipc_chunks.append(chunk)

ipc_df = pd.concat(ipc_chunks, ignore_index=True)
ipc_df['main_class'] = (
    ipc_df['section'].fillna('').astype(str) +
    ipc_df['ipc_class'].fillna(0).astype(str)
)
ipc_df['sub_class'] = ipc_df['main_class'] + ipc_df['subclass'].fillna('').astype(str)

ipc_rows = []
for pid, grp in ipc_df.groupby('patent_id'):
    row = {
        'patent_id': pid,
        'MCD': grp['main_class'].nunique(),
        'SCD': grp['sub_class'].nunique()
    }
    secs = set(grp['section'].dropna().astype(str).str.upper())
    for s in 'ABCDEFGH':
        row[f'TD_{s}'] = int(s in secs)
    ipc_rows.append(row)

ipc_feat = pd.DataFrame(ipc_rows)
ipc_feat.to_parquet(f'{DATA_DIR}/ckpt_ipc.parquet', index=False)
print(f'IPC: {len(ipc_feat):,}건  →  ckpt_ipc.parquet 저장')
del ipc_chunks, ipc_df; gc.collect()

---
## Step 6. Claims 피처 (CTS / PTS / CTC)

`g_claims_YYYY.tsv` 파일이 `../src/` 에 있으면 연도별로 처리.  
없으면 `num_claims` (g_patent.tsv)로 CTS를 대체하고 PTS / CTC = NaN.

In [ ]:
# [Cell 13]
# 사용법: TSV 올리고 실행 → 처리완료 → TSV 삭제 → 다음 배치 올리고 반복
CLAIMS_DIR = Path('./patentsview_data')
CLAIMS_CKPT_DIR = Path(DATA_DIR) / 'claims_by_year'
CLAIMS_CKPT_DIR.mkdir(exist_ok=True)

tsv_files = sorted(CLAIMS_DIR.glob('g_claims_*.tsv'))
print(f'발견된 TSV: {len(tsv_files)}개 → {[f.name for f in tsv_files]}')

for cfile in tsv_files:
    year = cfile.stem.split('_')[-1]
    ckpt_path = CLAIMS_CKPT_DIR / f'claims_{year}.parquet'

    if ckpt_path.exists():
        print(f'  {year}: 이미 처리됨 (스킵)')
        continue

    print(f'  {year} 처리 중...', flush=True)
    df = pd.read_csv(cfile, sep='\t', usecols=['patent_id', 'claim_text', 'dependent'], low_memory=False)
    df['patent_id'] = pd.to_numeric(df['patent_id'], errors='coerce')
    df = df[df['patent_id'].isin(all_pids)].copy()

    if len(df) == 0:
        print(f'  {year}: 매칭 특허 0건')
    else:
        df['wc'] = df['claim_text'].fillna('').str.split().str.len()
        year_feat = pd.concat([
            df.groupby('patent_id').size().rename('CTS'),
            df[df['dependent'].isna()].groupby('patent_id').size().rename('PTS'),
            df.groupby('patent_id')['wc'].sum().rename('CTC'),
        ], axis=1).reset_index()
        year_feat.to_parquet(ckpt_path, index=False)
        print(f'  {year} 완료: {len(year_feat):,}건 → claims_{year}.parquet 저장')
        del df, year_feat; gc.collect()

year_ckpts = sorted(CLAIMS_CKPT_DIR.glob('claims_*.parquet'))
done_years = [p.stem.replace('claims_', '') for p in year_ckpts]
print(f'\n완료된 연도 ({len(done_years)}개): {done_years}')

In [ ]:
# [Cell 14]
year_ckpts = sorted(CLAIMS_CKPT_DIR.glob('claims_*.parquet'))
print(f'합치기: {len(year_ckpts)}개 연도')

if year_ckpts:
    claims_all = pd.concat([pd.read_parquet(p) for p in year_ckpts], ignore_index=True)
    claims_feat = (
        claims_all.groupby('patent_id')
        .agg({'CTS': 'sum', 'PTS': 'sum', 'CTC': 'sum'})
        .reset_index()
    )
    claims_feat.to_parquet(f'{DATA_DIR}/ckpt_claims.parquet', index=False)
    print(f'Claims 통합 완료: {len(claims_feat):,}건  →  ckpt_claims.parquet 저장')
    del claims_all; gc.collect()
else:
    print('처리된 연도 없음 → num_claims로 CTS 대체')
    claims_feat = pat_util[['patent_id', 'num_claims']].rename(columns={'num_claims': 'CTS'}).copy()
    claims_feat['PTS'] = np.nan
    claims_feat['CTC'] = np.nan

---
## Step 7. 특허 피처 통합 + 소송 플래그

In [ ]:
# [Cell 15]
print('특허 피처 통합 중...', flush=True)

feat = pat_util[['patent_id', 'grant_date']].copy()

for df_, name in [
    (pk_df,       'PK'),
    (sk_df,       'SK'),
    (pps_df,      'PPS'),
    (et_df,       'ET'),
    (ie_df,       'IE'),
    (fie_df,      'FIE'),
    (abs_df,      'ABS'),
    (cd_df,       'CD'),
    (ic_df,       'IC'),
    (ipc_feat,    'IPC'),
    (claims_feat, 'Claims'),
]:
    feat = feat.merge(df_, on='patent_id', how='left')
    print(f'  + {name:8s} → {len(feat):,}건')

feat = feat.merge(patent_company, on='patent_id', how='left')
feat['is_litigated'] = feat['patent_id'].isin(litigated_set_int).astype(np.int8)

print(f'\n통합 완료: {len(feat):,}건 / {len(feat.columns)}컬럼')
print(f'소송: {feat["is_litigated"].sum():,}건 / 비소송: {(feat["is_litigated"]==0).sum():,}건')

---
## Step 8. 결측 처리

In [ ]:
# [Cell 16]
FILL_ZERO   = ['PK', 'SK', 'PPS', 'MCD', 'SCD',
               'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H',
               'CTS', 'PTS', 'CTC']
FILL_MEDIAN = ['ABS', 'IE', 'FIE', 'ET', 'CD', 'IC']

for col in FILL_ZERO:
    if col in feat.columns:
        feat[col] = feat[col].fillna(0)
for col in FILL_MEDIAN:
    if col in feat.columns:
        feat[col] = feat[col].fillna(feat[col].median())

remaining_na = feat[PATENT_FEAT_COLS].isna().sum()
print('결측 처리 완료 — 잔여 결측:')
print(remaining_na[remaining_na > 0] if remaining_na.any() else '  없음')

---
## Step 9. 저장

In [ ]:
# [Cell 17]
save_cols = ['patent_id', 'org_norm', 'grant_date', 'is_litigated'] + PATENT_FEAT_COLS
save_cols = [c for c in save_cols if c in feat.columns]

feat[save_cols].to_parquet(f'{DATA_DIR}/patent_features.parquet', index=False)
print(f'patent_features.parquet: {len(feat):,}건 / {len(save_cols)}컬럼')

feat[['patent_id', 'org_norm']].to_parquet(f'{DATA_DIR}/patent_company_map.parquet', index=False)
print(f'patent_company_map.parquet: {len(feat):,}건')

---
## Step 10. 커버리지 리포트

In [ ]:
# [Cell 18]
cov_rows = []
for col in PATENT_FEAT_COLS:
    if col not in feat.columns:
        continue
    cov_rows.append({
        'Feature':   col,
        'All':       feat[col].notna().mean(),
        'Litigated': feat.loc[feat['is_litigated']==1, col].notna().mean(),
        'Non-Lit':   feat.loc[feat['is_litigated']==0, col].notna().mean(),
    })

cov_df = pd.DataFrame(cov_rows).set_index('Feature')
print('=== 피처 커버리지 ===')
print(cov_df.round(3).to_string())
cov_df.to_csv(f'{DATA_DIR}/feature_coverage.csv', encoding='utf-8-sig')

In [ ]:
# [Cell 19]
feat_present = [c for c in PATENT_FEAT_COLS if c in feat.columns]
comp = feat.groupby('is_litigated')[feat_present].mean().T
comp.columns = ['Non-Lit (0)', 'Litigated (1)']
comp['L/NL ratio'] = (comp['Litigated (1)'] / (comp['Non-Lit (0)'] + 1e-8)).round(2)

print('=== 소송 vs 비소송 피처 평균 비교 ===')
print(comp.sort_values('L/NL ratio', ascending=False).round(3).to_string())

print('\n=== 저장 파일 ===')
for f in sorted(Path(DATA_DIR).glob('*')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:45s}  {size_mb:8.1f} MB')

---
## Step 11. ML Baseline — Model 1: L vs NL 소송 예측 (LightGBM + Optuna)

TCP·LI 제거 후 **22개 피처**만으로 소송 예측 성능 검증.  
GNN 추가 전 baseline 확인 목적.

In [ ]:
# [Cell 20]
import pickle
import lightgbm as lgb
import optuna
import shap
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42

def _check_lgb_gpu():
    try:
        m = lgb.LGBMClassifier(device='gpu', n_estimators=5, verbose=-1)
        m.fit(np.random.rand(50, 3), np.random.randint(0, 2, 50))
        return True
    except:
        return False

USE_GPU = _check_lgb_gpu()
DEVICE  = 'gpu' if USE_GPU else 'cpu'
print(f'LightGBM device: {DEVICE}')
print(f'피처 수: {len(PATENT_FEAT_COLS)}개')

In [ ]:
# [Cell 21]


In [ ]:
# [Cell 22]
feat_cols = PATENT_FEAT_COLS  # 22개

M1_FEAT_SETS = {
    'all_22':  feat_cols,
    'no_td_d': [f for f in feat_cols if f != 'TD_D'],  # TD_D: ~0.8% non-zero
}

print('=== M1 피처셋 ===')
for name, fs in M1_FEAT_SETS.items():
    print(f'  {name:10s} ({len(fs):2d}개): {fs}')

In [ ]:
# [Cell 23]
def _lgb_params(trial, extra=None):
    p = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500, step=100),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'device': DEVICE, 'n_jobs': 1, 'random_state': RANDOM_STATE, 'verbose': -1,
    }
    if extra:
        p.update(extra)
    return p

def make_objective_m1(feat_set, df_train):
    X = df_train[feat_set].values
    y = df_train['is_litigated'].values
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    def objective(trial):
        model = lgb.LGBMClassifier(**_lgb_params(trial))
        aucs = []
        for tr_idx, val_idx in cv.split(X, y):
            model.fit(X[tr_idx], y[tr_idx])
            proba = model.predict_proba(X[val_idx])[:, 1]
            aucs.append(roc_auc_score(y[val_idx], proba))
        return np.mean(aucs)
    return objective

print('Optuna 목적함수 정의 완료')

In [ ]:
# [Cell 24]
from sklearn.metrics import matthews_corrcoef
import os

N_TRIALS = 40
TIMEOUT  = 300
M1_CKPT  = f'{DATA_DIR}/m1_results.pkl'

def threshold_metrics(y_true, proba, threshold=None):
    if threshold is None:
        best_gm, best_thr = 0, 0.5
        for thr in np.linspace(0.01, 0.99, 99):
            pred = (proba >= thr).astype(int)
            tp = ((pred==1)&(y_true==1)).sum()
            fp = ((pred==1)&(y_true==0)).sum()
            fn = ((pred==0)&(y_true==1)).sum()
            tn = ((pred==0)&(y_true==0)).sum()
            r = tp/(tp+fn+1e-9); sp = tn/(tn+fp+1e-9)
            gm = np.sqrt(r * sp)
            if gm > best_gm: best_gm, best_thr = gm, thr
        threshold = best_thr
    pred = (proba >= threshold).astype(int)
    tp = ((pred==1)&(y_true==1)).sum(); fp = ((pred==1)&(y_true==0)).sum()
    fn = ((pred==0)&(y_true==1)).sum(); tn = ((pred==0)&(y_true==0)).sum()
    p = tp/(tp+fp+1e-9); r = tp/(tp+fn+1e-9); sp = tn/(tn+fp+1e-9)
    return {'threshold': threshold,
            'BA': (r+sp)/2, 'Prec': p, 'Rec': r,
            'F1': 2*p*r/(p+r+1e-9), 'F2': 5*p*r/(4*p+r+1e-9),
            'Gmean': np.sqrt(r*sp), 'MCC': matthews_corrcoef(y_true, pred)}

# ── 모델 로드 or 튜닝 ──────────────────────────────────────────────────────
if os.path.exists(M1_CKPT):
    with open(M1_CKPT, 'rb') as f:
        m1_models = pickle.load(f)
    print(f'✓ 모델 로드 ({list(m1_models.keys())})  → predict_proba 재계산')
else:
    m1_models = {}
    for set_name, fs in M1_FEAT_SETS.items():
        print(f'\n[Model1] {set_name} ({len(fs)}개 피처) 튜닝 중...')
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
        )
        study.optimize(make_objective_m1(fs, train_m1),
                       n_trials=N_TRIALS, timeout=TIMEOUT, show_progress_bar=True)
        best_params = study.best_params.copy()
        best_params.update({'device': DEVICE, 'n_jobs': -1,
                            'random_state': RANDOM_STATE, 'verbose': -1})
        model = lgb.LGBMClassifier(**best_params)
        model.fit(train_m1[fs], train_m1['is_litigated'])
        m1_models[set_name] = {
            'model': model, 'feat_set': fs,
            'best_params': best_params, 'cv_auc': study.best_value,
        }
        print(f'  완료  CV AUC={study.best_value:.4f}')

    with open(M1_CKPT, 'wb') as f:
        pickle.dump(m1_models, f)
    print(f'\n✓ 모델 저장 → {M1_CKPT}')

# ── predict_proba + 지표 계산 (항상 실행) ─────────────────────────────────
m1_results = {}
y_te = test_m1['is_litigated'].values

for set_name, m in m1_models.items():
    fs    = m['feat_set']
    model = m['model']
    proba = model.predict_proba(test_m1[fs])[:, 1]
    tr_proba = model.predict_proba(train_m1[fs])[:, 1]
    best_thr = threshold_metrics(train_m1['is_litigated'].values, tr_proba)['threshold']
    te_m     = threshold_metrics(y_te, proba, threshold=best_thr)
    te_m_05  = threshold_metrics(y_te, proba, threshold=0.5)
    m1_results[set_name] = {
        **m,
        'test_auc': roc_auc_score(y_te, proba),
        'test_ap':  average_precision_score(y_te, proba),
        'test_proba': proba, 'y_te': y_te, 'threshold': best_thr,
        **{f'test_{k}': v for k, v in te_m.items() if k != 'threshold'},
        **{f'test05_{k}': v for k, v in te_m_05.items() if k != 'threshold'},
    }

# ── 비교표 ─────────────────────────────────────────────────────────────────
W = 105
HDR = f'{"모델":30s}  {"Thr":5s}  {"BA":6s}  {"Prec":6s}  {"Rec":6s}  {"F1":6s}  {"F2":6s}  {"Gmean":6s}  {"MCC":6s}'
print('\n' + '='*W)
print('  Model 1 — L vs NL 비교  (AUC/AP는 threshold 무관)')
print('='*W)
print(HDR); print('-'*W)
print(f'{"논문(기존)":30s}  {"0.50":5s}  0.700   0.016   0.629   0.031   0.073   0.697   0.073')
print('-'*W)
for name, r in m1_results.items():
    thr = r['threshold']
    print(f'{"LGB-"+name+" (Gmean-opt)":30s}  {thr:.2f}   '
          f'{r["test_BA"]:.4f}  {r["test_Prec"]:.4f}  {r["test_Rec"]:.4f}  '
          f'{r["test_F1"]:.4f}  {r["test_F2"]:.4f}  {r["test_Gmean"]:.4f}  {r["test_MCC"]:.4f}')
    print(f'{"LGB-"+name+" (thr=0.50)":30s}  {"0.50":5s}  '
          f'{r["test05_BA"]:.4f}  {r["test05_Prec"]:.4f}  {r["test05_Rec"]:.4f}  '
          f'{r["test05_F1"]:.4f}  {r["test05_F2"]:.4f}  {r["test05_Gmean"]:.4f}  {r["test05_MCC"]:.4f}')
print('='*W)
print(f'  AUC / AP: ', '  '.join(
    f'{n}: AUC={r["test_auc"]:.4f} AP={r["test_ap"]:.4f}'
    for n, r in m1_results.items()))


In [ ]:
# [Cell 25]
best_m1_name = max(m1_results, key=lambda k: m1_results[k]['test_auc'])
best_m1 = m1_results[best_m1_name]
print(f'Best 피처셋: {best_m1_name}  (Test AUC {best_m1["test_auc"]:.4f})')

shap_sample = train_m1[best_m1['feat_set']].sample(min(2000, len(train_m1)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(best_m1['model'])
shap_vals = explainer.shap_values(shap_sample)
sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

shap_imp = pd.Series(
    np.abs(sv).mean(axis=0),
    index=best_m1['feat_set']
).sort_values(ascending=False)

print('\n=== SHAP Feature Importance ===')
print(shap_imp.round(4).to_string())

In [ ]:
# [Cell 26]
import datetime
today = datetime.date.today().strftime('%Y%m%d')

best_m1_name = max(m1_results, key=lambda k: m1_results[k]['test_auc'])
best_m1 = m1_results[best_m1_name]

save_m1 = {
    'name':        f'model1_final_{today}',
    'feat_cols':   best_m1['feat_set'],
    'model':       best_m1['model'],
    'best_params': best_m1['best_params'],
    'cv_auc':      best_m1['cv_auc'],
    'test_auc':    best_m1['test_auc'],
    'test_ap':     best_m1['test_ap'],
}
with open(f'{DATA_DIR}/model1_final.pkl', 'wb') as f:
    pickle.dump(save_m1, f)
print(f'모델 저장: {DATA_DIR}/model1_final.pkl')
print(f'  이름: {save_m1["name"]}')
print(f'  피처: {len(save_m1["feat_cols"])}개')
print(f'  Test AUC: {save_m1["test_auc"]:.4f}')
print(f'  Test AP:  {save_m1["test_ap"]:.4f}')

---
## Step 13. GNN Baseline — GraphSAGE + LightGBM

citation 그래프 구조를 활용해 ML baseline 대비 성능 개선 측정.  
전체 5.5M 노드 semi-supervised: message passing은 전체, loss는 train 레이블만.

In [78]:
# [Cell 27]
# pyg-lib는 GLIBC 버전 문제로 제거, torch-scatter/sparse만 사용
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
# 이미 설치된 경우 pyg-lib가 있으면 제거 필요
# !pip uninstall pyg-lib -y

ERROR: Could not find a version that satisfies the requirement pyg-lib (from versions: none)
ERROR: No matching distribution found for pyg-lib
Note: you may need to restart the kernel to use updated packages.


---
## ⚡ 커널 재시작 후 실행 순서

커널을 재시작했다면 아래 두 셀만 순서대로 실행하면 모든 모델 (SAGE / GAT / HeteroSAGE / HeteroGAT 등) 바로 실행 가능

1. **Cell 1** — 기본 import (`pandas`, `numpy`, 경로 설정)
2. **Cell 28 (바로 아래)** — GNN 전체 환경 (`torch`, `PyG`, 피처/마스크 로드, `focal_loss`, `threshold_metrics`, LightGBM 설정)

> Cell 2~27은 데이터 전처리 / pip install 셀이므로 재실행 불필요

In [5]:
# [Cell 28] GNN 환경 설정 — 커널 재시작 후 Cell 1 → 이 셀 순서로 실행
import os, gc, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef

if 'SRC_DIR' not in dir():
    SRC_DIR  = '../src'
if 'DATA_DIR' not in dir():
    DATA_DIR = './data_gnn'
if 'PATENT_FEAT_COLS' not in dir():
    PATENT_FEAT_COLS = [
        'MCD', 'SCD',
        'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H',
        'ABS', 'CTS', 'PTS', 'CTC',
        'PK', 'SK', 'PPS',
        'CD', 'IC', 'IE', 'FIE', 'ET',
    ]

RANDOM_STATE = 42

if 'train_m1' not in dir() or 'test_m1' not in dir():
    _train_pids = pd.read_parquet(f'{DATA_DIR}/split_train_pids.parquet')['patent_id'].tolist()
    _test_df    = pd.read_parquet(f'{DATA_DIR}/split_test.parquet')
    _feat = pd.read_parquet(f'{DATA_DIR}/patent_features.parquet',
                             columns=['patent_id'] + PATENT_FEAT_COLS + ['is_litigated'])
    train_m1 = _feat[_feat['patent_id'].isin(_train_pids)].reset_index(drop=True)
    test_m1  = _feat[_feat['patent_id'].isin(_test_df['patent_id'])].reset_index(drop=True)
    del _feat, _test_df
    print(f'  ✓ split 복원: train {len(train_m1):,} / test {len(test_m1):,}')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GATConv, HeteroConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.data import Data, HeteroData

GNN_CKPT_DIR = Path(DATA_DIR) / 'gnn'
GNN_CKPT_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}'
          f'  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

# 노드 피처 로드
df_gnn = pd.read_parquet(f'{DATA_DIR}/patent_features.parquet')
df_gnn = df_gnn.reset_index(drop=True)
pid_to_idx = {int(pid): i for i, pid in enumerate(df_gnn['patent_id'].values)}
N = len(df_gnn)

x = torch.tensor(df_gnn[PATENT_FEAT_COLS].values, dtype=torch.float32)
y = torch.tensor(df_gnn['is_litigated'].values,    dtype=torch.long)

train_pids = set(train_m1['patent_id'].values.tolist())
test_pids  = set(test_m1['patent_id'].values.tolist())

train_mask = torch.zeros(N, dtype=torch.bool)
test_mask  = torch.zeros(N, dtype=torch.bool)
for pid, idx in pid_to_idx.items():
    if pid in train_pids:  train_mask[idx] = True
    if pid in test_pids:   test_mask[idx]  = True

print(f'노드 수: {N:,}  |  피처: {x.shape[1]}')
print(f'train mask: {train_mask.sum():,}  test mask: {test_mask.sum():,}')
L_in_train = y[train_mask].sum().item()
print(f'train L={L_in_train:,} / NL={train_mask.sum().item()-L_in_train:,}')

# ── 유틸 함수 ──────────────────────────────────────────────────────────────
def focal_loss(logits, labels, gamma=2.0, pos_weight=1.0):
    bce = F.binary_cross_entropy_with_logits(
        logits, labels,
        pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=logits.device),
        reduction='none',
    )
    pt = torch.exp(-bce)
    return (((1 - pt) ** gamma) * bce).mean()

def eval_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    tp = ((pred==1)&(y_true==1)).sum(); fp = ((pred==1)&(y_true==0)).sum()
    fn = ((pred==0)&(y_true==1)).sum(); tn = ((pred==0)&(y_true==0)).sum()
    p = tp/(tp+fp+1e-9); r = tp/(tp+fn+1e-9); sp = tn/(tn+fp+1e-9)
    return {
        'AUC':   roc_auc_score(y_true, proba),
        'AP':    average_precision_score(y_true, proba),
        'Prec':  p, 'Rec': r,
        'F1':    2*p*r/(p+r+1e-9),
        'Gmean': np.sqrt(r*sp),
        'MCC':   matthews_corrcoef(y_true, pred),
    }

# ── LightGBM 설정 ──────────────────────────────────────────────────────────
import lightgbm as lgb
try:
    lgb.LGBMClassifier(device='gpu', n_estimators=5, verbose=-1).fit(
        np.random.rand(50, 3), np.random.randint(0, 2, 50))
    DEVICE = 'gpu'
except:
    DEVICE = 'cpu'
print(f'LightGBM device: {DEVICE}')

# ── m1_results (LGB 베이스라인 하이퍼파라미터) ────────────────────────────
try:
    with open(f'{DATA_DIR}/m1_results.pkl', 'rb') as f:
        m1_results = pickle.load(f)
    print(f'✓ m1_results 로드: {list(m1_results.keys())}')
except FileNotFoundError:
    m1_results = None
    print('⚠ m1_results 없음 (Cell 24 먼저 실행 필요)')

print('\n✓ GNN 환경 준비 완료')


  ✓ split 복원: train 53,766 / test 1,113,677
device: cuda
  GPU: NVIDIA RTX A6000  (51.0 GB)
노드 수: 5,568,384  |  피처: 22
train mask: 53,766  test mask: 1,113,677
train L=26,883 / NL=26,883
LightGBM device: gpu
✓ m1_results 로드: ['all_22', 'no_td_d']

✓ GNN 환경 준비 완료


In [74]:
# [Cell 29]
EDGE_CKPT = GNN_CKPT_DIR / 'edge_index.pt'

if EDGE_CKPT.exists():
    edge_index = torch.load(EDGE_CKPT, weights_only=True)
    print(f'✓ edge_index 로드: {edge_index.shape[1]:,}개 엣지')
else:
    # pandas로 컬럼명 읽기 (따옴표 처리 자동)
    cit_cols = pd.read_csv(
        f'{SRC_DIR}/g_us_patent_citation.tsv', sep='\t', nrows=0
    ).columns.tolist()
    print('citation 컬럼:', cit_cols)

    cit_id_col = next((c for c in cit_cols
                       if 'citation' in c.lower() and 'patent' in c.lower()
                       and c != 'patent_id'), None)
    if cit_id_col is None:
        cit_id_col = [c for c in cit_cols if c != 'patent_id' and 'id' in c.lower()][0]
    print(f'  citing: patent_id  →  cited: {cit_id_col}')

    pid_series = pd.Series(pid_to_idx)  # pid → node_idx (벡터화 map용)

    src_parts, dst_parts = [], []
    for i, chunk in enumerate(pd.read_csv(
        f'{SRC_DIR}/g_us_patent_citation.tsv', sep='\t',
        usecols=['patent_id', cit_id_col], low_memory=False, chunksize=300_000
    )):
        chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
        chunk[cit_id_col]  = pd.to_numeric(chunk[cit_id_col],  errors='coerce')
        chunk = chunk.dropna().astype(int)

        s_idx = chunk['patent_id'].map(pid_series)
        d_idx = chunk[cit_id_col].map(pid_series)
        valid = s_idx.notna() & d_idx.notna()
        s = s_idx[valid].astype(int).values
        d = d_idx[valid].astype(int).values

        src_parts.append(np.concatenate([s, d]))   # 양방향
        dst_parts.append(np.concatenate([d, s]))

        if (i + 1) % 10 == 0:
            print(f'  chunk {i+1:3d}  누적 엣지: {sum(len(x) for x in src_parts):,}')

    src_arr = np.concatenate(src_parts)
    dst_arr = np.concatenate(dst_parts)
    edge_index = torch.tensor([src_arr, dst_arr], dtype=torch.long)
    torch.save(edge_index, EDGE_CKPT)
    print(f'✓ edge_index 저장: {edge_index.shape[1]:,}개 엣지 (양방향)')
    del src_parts, dst_parts, src_arr, dst_arr, pid_series

data = Data(x=x, edge_index=edge_index, y=y)
data.train_mask = train_mask
data.test_mask  = test_mask
print(f'\nPyG Data: {data}')
print(f'  엣지 밀도: {edge_index.shape[1]/N:.1f} edges/node 평균')

citation 컬럼: ['patent_id', 'citation_sequence', 'citation_patent_id', 'citation_date', 'record_name', 'wipo_kind', 'citation_category']
  citing: patent_id  →  cited: citation_patent_id
  chunk  10  누적 엣지: 0
  chunk  20  누적 엣지: 0
  chunk  30  누적 엣지: 0
  chunk  40  누적 엣지: 0
  chunk  50  누적 엣지: 0
  chunk  60  누적 엣지: 0
  chunk  70  누적 엣지: 0
  chunk  80  누적 엣지: 0
  chunk  90  누적 엣지: 0
  chunk 100  누적 엣지: 0
  chunk 110  누적 엣지: 0
  chunk 120  누적 엣지: 0
  chunk 130  누적 엣지: 0
  chunk 140  누적 엣지: 0
  chunk 150  누적 엣지: 0
  chunk 160  누적 엣지: 0
  chunk 170  누적 엣지: 802,990
  chunk 180  누적 엣지: 3,715,480
  chunk 190  누적 엣지: 7,605,184
  chunk 200  누적 엣지: 12,064,274
  chunk 210  누적 엣지: 16,876,862
  chunk 220  누적 엣지: 21,886,406
  chunk 230  누적 엣지: 27,052,114
  chunk 240  누적 엣지: 32,318,280
  chunk 250  누적 엣지: 37,581,360
  chunk 260  누적 엣지: 42,886,728
  chunk 270  누적 엣지: 48,335,536
  chunk 280  누적 엣지: 53,824,590
  chunk 290  누적 엣지: 59,345,238
  chunk 300  누적 엣지: 64,870,242
  chunk 310  누적 엣지: 70,375,806
  

In [77]:
# [Cell 30]
# Cell 28 — GraphSAGE 모델 + 학습 (체크포인트)
# 커널 재시작 후 바로 실행 가능하도록 경로 재정의
if 'GNN_CKPT_DIR' not in dir():
    GNN_CKPT_DIR = Path(DATA_DIR) / 'gnn'
    GNN_CKPT_DIR.mkdir(exist_ok=True)
GNN_MODEL_CKPT = GNN_CKPT_DIR / 'graphsage.pt'

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_c, hidden_c=128, emb_c=64):
        super().__init__()
        self.conv1 = SAGEConv(in_c, hidden_c)
        self.conv2 = SAGEConv(hidden_c, emb_c)
        self.head  = torch.nn.Linear(emb_c, 1)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.3, training=self.training)
        return self.conv2(x, edge_index)

    def forward(self, x, edge_index):
        return self.head(self.encode(x, edge_index)).squeeze(-1)

if GNN_MODEL_CKPT.exists():
    model_gnn = GraphSAGE(in_c=len(PATENT_FEAT_COLS)).to(device)
    model_gnn.load_state_dict(torch.load(GNN_MODEL_CKPT, map_location=device))
    model_gnn.eval()
    print(f'✓ GraphSAGE 로드: {GNN_MODEL_CKPT}')
else:
    n_pos = int(y[train_mask].sum())
    n_neg = int(train_mask.sum()) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
    criterion  = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    loader = NeighborLoader(
        data, num_neighbors=[25, 10], batch_size=512,
        input_nodes=data.train_mask, shuffle=True,
    )

    model_gnn = GraphSAGE(in_c=len(PATENT_FEAT_COLS)).to(device)
    optimizer = torch.optim.Adam(model_gnn.parameters(), lr=1e-3, weight_decay=1e-5)

    EPOCHS = 50
    print(f'학습 시작 (epochs={EPOCHS}, device={device})')
    for epoch in range(1, EPOCHS + 1):
        model_gnn.train()
        total_loss, n_batch = 0.0, 0
        for batch in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out  = model_gnn(batch.x, batch.edge_index)[:batch.batch_size]
            lbl  = batch.y[:batch.batch_size].float()
            loss = criterion(out, lbl)
            loss.backward()
            optimizer.step()
            total_loss += loss.item(); n_batch += 1
        print(f'  Epoch {epoch:3d}/{EPOCHS}  loss={total_loss/n_batch:.4f}', flush=True)

    torch.save(model_gnn.state_dict(), GNN_MODEL_CKPT)
    print(f'✓ 모델 저장: {GNN_MODEL_CKPT}')


학습 시작 (epochs=50, device=cuda)


ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [ ]:
# [Cell 31]
# 커널 재시작 후 바로 실행 가능하도록 복원
from sklearn.metrics import matthews_corrcoef, roc_auc_score, average_precision_score

def threshold_metrics(y_true, proba, threshold=None):
    if threshold is None:
        best_gm, best_thr = 0, 0.5
        for thr in np.linspace(0.01, 0.99, 99):
            pred = (proba >= thr).astype(int)
            tp = ((pred==1)&(y_true==1)).sum()
            fp = ((pred==1)&(y_true==0)).sum()
            fn = ((pred==0)&(y_true==1)).sum()
            tn = ((pred==0)&(y_true==0)).sum()
            r = tp/(tp+fn+1e-9); sp = tn/(tn+fp+1e-9)
            gm = np.sqrt(r * sp)
            if gm > best_gm: best_gm, best_thr = gm, thr
        threshold = best_thr
    pred = (proba >= threshold).astype(int)
    tp = ((pred==1)&(y_true==1)).sum(); fp = ((pred==1)&(y_true==0)).sum()
    fn = ((pred==0)&(y_true==1)).sum(); tn = ((pred==0)&(y_true==0)).sum()
    p = tp/(tp+fp+1e-9); r = tp/(tp+fn+1e-9); sp = tn/(tn+fp+1e-9)
    return {'threshold': threshold,
            'BA': (r+sp)/2, 'Prec': p, 'Rec': r,
            'F1': 2*p*r/(p+r+1e-9), 'F2': 5*p*r/(4*p+r+1e-9),
            'Gmean': np.sqrt(r*sp), 'MCC': matthews_corrcoef(y_true, pred)}

RANDOM_STATE = 42

if 'm1_results' not in dir():
    with open(f'{DATA_DIR}/m1_results.pkl', 'rb') as f:
        m1_results = pickle.load(f)
    print(f'✓ m1_results 로드: {list(m1_results.keys())}')

if 'DEVICE' not in dir():
    import lightgbm as lgb
    try:
        lgb.LGBMClassifier(device='gpu', n_estimators=5, verbose=-1).fit(
            np.random.rand(50, 3), np.random.randint(0, 2, 50))
        DEVICE = 'gpu'
    except:
        DEVICE = 'cpu'
    print(f'LightGBM device: {DEVICE}')

EMB_CKPT = GNN_CKPT_DIR / 'embeddings.pt'

if EMB_CKPT.exists():
    emb_all = torch.load(EMB_CKPT, map_location='cpu').numpy()
    print(f'✓ 임베딩 로드: {emb_all.shape}')
else:
    inf_loader = NeighborLoader(
        data, num_neighbors=[25, 10], batch_size=1024,
        input_nodes=torch.arange(N), shuffle=False,
    )
    model_gnn.eval()
    emb_list = []
    with torch.no_grad():
        for batch in inf_loader:
            batch = batch.to(device)
            emb   = model_gnn.encode(batch.x, batch.edge_index)[:batch.batch_size]
            emb_list.append(emb.cpu())
    emb_all = torch.cat(emb_list, dim=0).numpy()
    torch.save(torch.tensor(emb_all), EMB_CKPT)
    print(f'✓ 임베딩 저장: {emb_all.shape}')

EMB_COLS = [f'gnn_{i}' for i in range(emb_all.shape[1])]
df_emb   = pd.DataFrame(emb_all, columns=EMB_COLS)
df_emb['patent_id'] = df_gnn['patent_id'].values

df_aug = df_gnn[['patent_id', 'is_litigated'] + PATENT_FEAT_COLS].merge(
    df_emb, on='patent_id', how='left')

FEAT_GNN = PATENT_FEAT_COLS + EMB_COLS

aug_train = df_aug[df_aug['patent_id'].isin(train_pids)].copy()
aug_test  = df_aug[df_aug['patent_id'].isin(test_pids)].copy()
print(f'aug_train: {len(aug_train):,}  aug_test: {len(aug_test):,}')

best_set  = max(m1_results, key=lambda k: m1_results[k]['test_auc'])
base_params = m1_results[best_set]['best_params'].copy()
base_params.update({'device': DEVICE, 'n_jobs': -1,
                    'random_state': RANDOM_STATE, 'verbose': -1})
lgb_gnn = lgb.LGBMClassifier(**base_params)
lgb_gnn.fit(aug_train[FEAT_GNN], aug_train['is_litigated'])

y_te     = aug_test['is_litigated'].values
proba_g  = lgb_gnn.predict_proba(aug_test[FEAT_GNN])[:, 1]
auc_g    = roc_auc_score(y_te, proba_g)
ap_g     = average_precision_score(y_te, proba_g)

tr_proba_g = lgb_gnn.predict_proba(aug_train[FEAT_GNN])[:, 1]
best_thr_g = threshold_metrics(aug_train['is_litigated'].values, tr_proba_g)['threshold']
te_m_g     = threshold_metrics(y_te, proba_g, threshold=best_thr_g)
te_m_g_05  = threshold_metrics(y_te, proba_g, threshold=0.5)

print(f'\nSAGE+LGB  AUC={auc_g:.4f}  AP={ap_g:.4f}')
print(f'  Gmean-opt: Thr={best_thr_g:.3f}  BA={te_m_g["BA"]:.4f}  F1={te_m_g["F1"]:.4f}  '
      f'Rec={te_m_g["Rec"]:.4f}  MCC={te_m_g["MCC"]:.4f}')
print(f'  thr=0.50:  BA={te_m_g_05["BA"]:.4f}  F1={te_m_g_05["F1"]:.4f}  '
      f'Rec={te_m_g_05["Rec"]:.4f}  MCC={te_m_g_05["MCC"]:.4f}')

# ── 최종 비교표 ────────────────────────────────────────────────────────────
best_ml = m1_results[best_set]
W = 100
print('\n' + '='*W)
print(f'{"모델":26s}  {"Thr":5s}  {"AUC":6s}  {"AP":6s}  {"Prec":6s}  {"Rec":6s}  {"F1":6s}  {"MCC":6s}  {"Gmean":6s}')
print('-'*W)
print(f'{"논문(기존)":26s}  {"0.50":5s}  {"":6s}  {"":6s}  {"0.016":6s}  {"0.629":6s}  {"0.031":6s}  {"0.073":6s}  {"0.697":6s}')
print('-'*W)
# LGB rows
thr_ml = best_ml['threshold']
print(f'{"LGB (22 feat, Gmean-opt)":26s}  {thr_ml:.2f}   '
      f'{best_ml["test_auc"]:.4f}  {best_ml["test_ap"]:.4f}  '
      f'{best_ml.get("test_Prec",float("nan")):.4f}  {best_ml.get("test_Rec",float("nan")):.4f}  '
      f'{best_ml.get("test_F1",float("nan")):.4f}  {best_ml.get("test_MCC",float("nan")):.4f}  '
      f'{best_ml.get("test_Gmean",float("nan")):.4f}')
print(f'{"LGB (22 feat, thr=0.50)":26s}  {"0.50":5s}  '
      f'{best_ml["test_auc"]:.4f}  {best_ml["test_ap"]:.4f}  '
      f'{best_ml.get("test05_Prec",float("nan")):.4f}  {best_ml.get("test05_Rec",float("nan")):.4f}  '
      f'{best_ml.get("test05_F1",float("nan")):.4f}  {best_ml.get("test05_MCC",float("nan")):.4f}  '
      f'{best_ml.get("test05_Gmean",float("nan")):.4f}')
# SAGE rows
print(f'{"SAGE+LGB (Gmean-opt)":26s}  {best_thr_g:.2f}   '
      f'{auc_g:.4f}  {ap_g:.4f}  '
      f'{te_m_g["Prec"]:.4f}  {te_m_g["Rec"]:.4f}  '
      f'{te_m_g["F1"]:.4f}  {te_m_g["MCC"]:.4f}  {te_m_g["Gmean"]:.4f}')
print(f'{"SAGE+LGB (thr=0.50)":26s}  {"0.50":5s}  '
      f'{auc_g:.4f}  {ap_g:.4f}  '
      f'{te_m_g_05["Prec"]:.4f}  {te_m_g_05["Rec"]:.4f}  '
      f'{te_m_g_05["F1"]:.4f}  {te_m_g_05["MCC"]:.4f}  {te_m_g_05["Gmean"]:.4f}')
print('='*W)


---
## Step 13-1: GraphSAGE (GATFraud 모델 구조 동일화)

GATFraud 성능 향상 원인 분리 실험:
- **GAT attention 자체의 효과** vs **3-layer + BN + Focal Loss + Residual 구조**
- GraphSAGE에 GATFraud와 동일한 구조(3-layer, BN, Residual, Focal Loss) 적용
- 두 모델의 차이 = SAGEConv vs GATConv (attention 유무)

In [8]:
# [Cell 32-1]
# SAGEFraud: GraphSAGE + 3-layer + BN + Residual + Focal Loss (GATFraud 구조 동일화)
SAGEF_MDL_CKPT = GNN_CKPT_DIR / 'model_sage_fraud.pt'
SAGEF_EMB_CKPT = GNN_CKPT_DIR / 'embeddings_sage_fraud.pt'

class SAGEFraud(torch.nn.Module):
    def __init__(self, in_channels, hidden=256, drop=0.3):
        super().__init__()
        self.drop  = drop
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.bn3   = nn.BatchNorm1d(hidden)
        self.clf   = nn.Linear(hidden, 1)

    def encode(self, x, edge_index):
        x1 = F.dropout(x,  p=self.drop, training=self.training)
        x1 = F.elu(self.bn1(self.conv1(x1, edge_index)))
        x2 = F.dropout(x1, p=self.drop, training=self.training)
        x2 = F.elu(self.bn2(self.conv2(x2, edge_index))) + x1   # residual
        x3 = F.dropout(x2, p=self.drop, training=self.training)
        x3 = F.elu(self.bn3(self.conv3(x3, edge_index)))
        return x3

    def forward(self, x, edge_index):
        return self.clf(self.encode(x, edge_index)).squeeze(-1)

feat_df_sf = pd.read_parquet(f'{DATA_DIR}/patent_features.parquet',
                              columns=['patent_id'] + PATENT_FEAT_COLS + ['is_litigated']).fillna(0)
x_sf = torch.tensor(feat_df_sf[PATENT_FEAT_COLS].values, dtype=torch.float32)
y_sf = torch.tensor(feat_df_sf['is_litigated'].values,   dtype=torch.long)

train_pids_sf = set(pd.read_parquet(f'{DATA_DIR}/split_train_pids.parquet')['patent_id'].tolist())
test_pids_sf  = set(pd.read_parquet(f'{DATA_DIR}/split_test.parquet')['patent_id'].tolist())
pid_list_sf   = feat_df_sf['patent_id'].tolist()
train_mask_sf = torch.tensor([p in train_pids_sf for p in pid_list_sf], dtype=torch.bool)
test_mask_sf  = torch.tensor([p in test_pids_sf  for p in pid_list_sf], dtype=torch.bool)

ei_sf   = torch.load(GNN_CKPT_DIR / 'edge_index.pt', weights_only=True)
data_sf = Data(x=x_sf, y=y_sf, edge_index=ei_sf,
               train_mask=train_mask_sf, test_mask=test_mask_sf)

_nn_sf       = [10, 10, 5]
train_idx_sf = data_sf.train_mask.nonzero(as_tuple=True)[0]
test_idx_sf  = data_sf.test_mask.nonzero(as_tuple=True)[0]
train_loader_sf = NeighborLoader(data_sf, num_neighbors=_nn_sf,
                                  batch_size=1024, input_nodes=train_idx_sf,
                                  shuffle=True, num_workers=8)
test_loader_sf  = NeighborLoader(data_sf, num_neighbors=_nn_sf,
                                  batch_size=2048, input_nodes=test_idx_sf,
                                  shuffle=False, num_workers=8)

model_sage_fraud = SAGEFraud(in_channels=len(PATENT_FEAT_COLS)).to(device)
print(f'파라미터 수: {sum(p.numel() for p in model_sage_fraud.parameters() if p.requires_grad):,}')

pos_sf = float((y_sf[train_mask_sf] == 0).sum() / y_sf[train_mask_sf].sum())

def train_epoch_sf(model, loader, opt):
    model.train()
    total = 0
    for batch in loader:
        batch = batch.to(device)
        n     = batch.batch_size
        opt.zero_grad()
        logits = model(batch.x, batch.edge_index)[:n]
        loss   = focal_loss(logits, batch.y[:n].float(), gamma=2.0, pos_weight=pos_sf)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_sf(model, loader):
    model.eval()
    probs, ys = [], []
    for batch in loader:
        batch = batch.to(device)
        n     = batch.batch_size
        probs.append(torch.sigmoid(model(batch.x, batch.edge_index)[:n]).cpu())
        ys.append(batch.y[:n].cpu())
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

if SAGEF_MDL_CKPT.exists():
    model_sage_fraud.load_state_dict(torch.load(SAGEF_MDL_CKPT, map_location=device, weights_only=True))
    print('✓ 체크포인트 로드')
else:
    opt_sf = torch.optim.Adam(model_sage_fraud.parameters(), lr=1e-3, weight_decay=1e-4)
    sch_sf = torch.optim.lr_scheduler.CosineAnnealingLR(opt_sf, T_max=30)
    best_auc_sf = 0
    for ep in range(1, 31):
        loss = train_epoch_sf(model_sage_fraud, train_loader_sf, opt_sf)
        sch_sf.step()
        if ep % 5 == 0:
            proba, ytmp = eval_sf(model_sage_fraud, test_loader_sf)
            auc = roc_auc_score(ytmp, proba)
            ap  = average_precision_score(ytmp, proba)
            print(f'Epoch {ep:02d} | loss={loss:.4f} | AUC={auc:.4f} | AP={ap:.4f}')
            if auc > best_auc_sf:
                best_auc_sf = auc
                torch.save(model_sage_fraud.state_dict(), SAGEF_MDL_CKPT)
    print(f'✓ 학습 완료 (best AUC={best_auc_sf:.4f})')
    model_sage_fraud.load_state_dict(torch.load(SAGEF_MDL_CKPT, map_location=device, weights_only=True))

if SAGEF_EMB_CKPT.exists():
    emb_sage_fraud = torch.load(SAGEF_EMB_CKPT, weights_only=True)
    print(f'✓ 임베딩 로드: {emb_sage_fraud.shape}')
else:
    all_loader_sf = NeighborLoader(data_sf, num_neighbors=_nn_sf,
                                    batch_size=4096, input_nodes=torch.arange(data_sf.num_nodes),
                                    shuffle=False, num_workers=8)
    model_sage_fraud.eval()
    emb_sage_fraud = torch.zeros(data_sf.num_nodes, 256)
    with torch.no_grad():
        for batch in all_loader_sf:
            batch = batch.to(device)
            n     = batch.batch_size
            emb_sage_fraud[batch.n_id[:n]] = model_sage_fraud.encode(batch.x, batch.edge_index)[:n].cpu()
    torch.save(emb_sage_fraud, SAGEF_EMB_CKPT)
    print(f'✓ 임베딩 저장: {emb_sage_fraud.shape}')


파라미터 수: 275,969
Epoch 05 | loss=0.1527 | AUC=0.7564 | AP=0.0222
Epoch 10 | loss=0.1481 | AUC=0.7886 | AP=0.0276
Epoch 15 | loss=0.1467 | AUC=0.7853 | AP=0.0281
Epoch 20 | loss=0.1443 | AUC=0.7928 | AP=0.0295
Epoch 25 | loss=0.1441 | AUC=0.7937 | AP=0.0301
Epoch 30 | loss=0.1437 | AUC=0.7956 | AP=0.0308
✓ 학습 완료 (best AUC=0.7956)
✓ 임베딩 저장: torch.Size([5568384, 256])


In [9]:
# [Cell 32-2]
# SAGEFraud + LGB
import lightgbm as lgb

# ── 커널 재시작 후 복원 ──
SF_EMB_CKPT = GNN_CKPT_DIR / 'embeddings_sage_fraud.pt'
emb_sage_fraud = torch.load(SF_EMB_CKPT, map_location='cpu')
print(f'✓ SAGEFraud 임베딩 로드: {emb_sage_fraud.shape}')

feat_df_sf = df_gnn.reset_index(drop=True)
y_sf = torch.tensor(df_gnn['is_litigated'].values, dtype=torch.long)
# ─────────────────────────

feat_np_sf = feat_df_sf[PATENT_FEAT_COLS].values
y_arr_sf   = y_sf.numpy()
emb_np_sf  = emb_sage_fraud.numpy()

SF_EMB_CKPT = GNN_CKPT_DIR / 'embeddings_sage_fraud.pt'
emb_sage_fraud = torch.load(SF_EMB_CKPT, map_location='cpu')

feat_df_sf = df_gnn.reset_index(drop=True)
y_sf = torch.tensor(df_gnn['is_litigated'].values, dtype=torch.long)

train_mask_sf = train_mask   # ← 이거 추가
test_mask_sf  = test_mask

✓ SAGEFraud 임베딩 로드: torch.Size([5568384, 256])


In [10]:
# [Cell 32-2]
# SAGEFraud + LGB
import lightgbm as lgb

feat_np_sf = feat_df_sf[PATENT_FEAT_COLS].values
y_arr_sf   = y_sf.numpy()
emb_np_sf  = emb_sage_fraud.numpy()

X_tr_sf = np.hstack([feat_np_sf[train_mask_sf.numpy()], emb_np_sf[train_mask_sf.numpy()]])
X_te_sf = np.hstack([feat_np_sf[test_mask_sf.numpy()],  emb_np_sf[test_mask_sf.numpy()]])
y_tr_sf = y_arr_sf[train_mask_sf.numpy()]
y_te_sf = y_arr_sf[test_mask_sf.numpy()]

if m1_results is not None:
    best_set    = max(m1_results, key=lambda k: m1_results[k]['test_auc'])
    base_params = m1_results[best_set]['best_params'].copy()
else:
    base_params = {'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 63,
                   'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8}

lgb_params_sf = {**base_params,
                  'scale_pos_weight': float((y_tr_sf == 0).sum() / y_tr_sf.sum()),
                  'device': DEVICE, 'n_jobs': -1,
                  'random_state': RANDOM_STATE, 'verbosity': -1}

lgb_sf = lgb.train(lgb_params_sf,
                    lgb.Dataset(X_tr_sf, y_tr_sf),
                    num_boost_round=500,
                    valid_sets=[lgb.Dataset(X_te_sf, y_te_sf)],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                                lgb.log_evaluation(period=-1)])

proba_sf = lgb_sf.predict(X_te_sf)
m_sf     = eval_metrics(y_te_sf, proba_sf, threshold=0.5)

print('SAGEFraud + LGB (thr=0.5)')
print(f'  AUC={m_sf["AUC"]:.4f}  AP={m_sf["AP"]:.4f}  '
      f'Prec={m_sf["Prec"]:.4f}  Rec={m_sf["Rec"]:.4f}  '
      f'F1={m_sf["F1"]:.4f}  Gmean={m_sf["Gmean"]:.4f}  MCC={m_sf["MCC"]:.4f}')


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


SAGEFraud + LGB (thr=0.5)
  AUC=0.8208  AP=0.0391  Prec=0.0168  Rec=0.7612  F1=0.0328  Gmean=0.7450  MCC=0.0852


---
## Step 14. GATFraud 모델 — 3-layer GAT + Focal Loss + BN + Residual

fraud-detection SOTA 구조 참조. GraphSAGE 대비 개선 포인트:

| 항목 | GraphSAGE (Step 13) | GATFraud (Step 14) |
|------|---|---|
| Aggregation | mean | **attention (GAT)** |
| Layers | 2 | **3** |
| Normalization | 없음 | **BatchNorm** |
| Residual | 없음 | **있음** |
| Loss | BCE + pos_weight | **Focal Loss (γ=2)** |
| Neighbor sampling | [25, 10] | **[10, 10, 5]** |
| LR scheduler | 없음 | **StepLR(step=10, γ=0.78)** |

Focal Loss: γ=2로 hard example(애매한 특허)에 집중, 쉬운 NL은 자동 다운웨이팅.

In [ ]:
# [Cell 32]
# Cell 30 — GATFraud 모델 + 학습 (3-layer GAT + BN + Residual + Focal Loss)
GAT_CKPT = GNN_CKPT_DIR / 'gatfraud.pt'

class GATFraud(torch.nn.Module):
    """3-layer GAT + BN + Residual (fraud-detection SOTA 구조)"""
    def __init__(self, in_c, hidden_c=32, emb_c=64, heads=8, dropout=0.3):
        super().__init__()
        self.drop = dropout

        self.conv1 = GATConv(in_c,           hidden_c, heads=heads, dropout=dropout, concat=True)
        self.bn1   = nn.BatchNorm1d(hidden_c * heads)

        self.conv2 = GATConv(hidden_c*heads, hidden_c, heads=heads, dropout=dropout, concat=True)
        self.bn2   = nn.BatchNorm1d(hidden_c * heads)

        self.conv3 = GATConv(hidden_c*heads, emb_c,    heads=1,    dropout=dropout, concat=False)
        self.bn3   = nn.BatchNorm1d(emb_c)

        self.head  = nn.Linear(emb_c, 1)

    def encode(self, x, edge_index):
        x1 = F.dropout(x,  p=self.drop, training=self.training)
        x1 = F.elu(self.bn1(self.conv1(x1, edge_index)))

        x2 = F.dropout(x1, p=self.drop, training=self.training)
        x2 = F.elu(self.bn2(self.conv2(x2, edge_index)))
        x2 = x2 + x1                                   # residual

        x3 = F.dropout(x2, p=self.drop, training=self.training)
        x3 = F.elu(self.bn3(self.conv3(x3, edge_index)))
        return x3

    def forward(self, x, edge_index):
        return self.head(self.encode(x, edge_index)).squeeze(-1)


def focal_loss(logits, labels, gamma=2.0, pos_weight=1.0):
    """Focal Loss — hard example mining for extreme class imbalance"""
    bce = F.binary_cross_entropy_with_logits(
        logits, labels,
        pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=logits.device),
        reduction='none'
    )
    p_t = torch.sigmoid(logits) * labels + (1 - torch.sigmoid(logits)) * (1 - labels)
    return ((1 - p_t) ** gamma * bce).mean()


if GAT_CKPT.exists():
    model_gat = GATFraud(in_c=len(PATENT_FEAT_COLS)).to(device)
    model_gat.load_state_dict(torch.load(GAT_CKPT, map_location=device))
    model_gat.eval()
    print(f'✓ GATFraud 로드: {GAT_CKPT}')
else:
    n_pos = int(y[train_mask].sum())
    n_neg = int(train_mask.sum()) - n_pos
    pw    = n_neg / n_pos
    print(f'pos_weight: {pw:.1f}  (train L={n_pos:,} / NL={n_neg:,})')

    loader_gat = NeighborLoader(
        data, num_neighbors=[10, 10, 5], batch_size=1024,
        input_nodes=data.train_mask, shuffle=True,
        num_workers=8, persistent_workers=True,
    )

    model_gat   = GATFraud(in_c=len(PATENT_FEAT_COLS)).to(device)
    opt_gat     = torch.optim.Adam(model_gat.parameters(), lr=1e-3, weight_decay=1e-5)
    sched_gat   = torch.optim.lr_scheduler.StepLR(opt_gat, step_size=10, gamma=0.78)

    EPOCHS = 50
    print(f'GATFraud 학습 시작 (epochs={EPOCHS}, device={device})')
    for epoch in range(1, EPOCHS + 1):
        model_gat.train()
        total_loss, n_batch = 0.0, 0
        for batch in loader_gat:
            batch = batch.to(device)
            opt_gat.zero_grad()
            out  = model_gat(batch.x, batch.edge_index)[:batch.batch_size]
            lbl  = batch.y[:batch.batch_size].float()
            loss = focal_loss(out, lbl, gamma=2.0, pos_weight=pw)
            loss.backward()
            opt_gat.step()
            total_loss += loss.item(); n_batch += 1
        sched_gat.step()
        print(f'  Epoch {epoch:3d}/{EPOCHS}  loss={total_loss/n_batch:.4f}'
              f'  lr={sched_gat.get_last_lr()[0]:.5f}', flush=True)

    torch.save(model_gat.state_dict(), GAT_CKPT)
    print(f'✓ GATFraud 저장: {GAT_CKPT}')


In [ ]:
# [Cell 33]
GAT_EMB_CKPT = GNN_CKPT_DIR / 'embeddings_gat.pt'

if GAT_EMB_CKPT.exists():
    emb_gat = torch.load(GAT_EMB_CKPT, map_location='cpu').numpy()
    print(f'✓ GAT 임베딩 로드: {emb_gat.shape}')
else:
    inf_loader_gat = NeighborLoader(
        data, num_neighbors=[10, 10, 5], batch_size=2048,
        input_nodes=torch.arange(N), shuffle=False,
        num_workers=8, persistent_workers=True,
    )
    model_gat.eval()
    emb_list_gat = []
    with torch.no_grad():
        for batch in inf_loader_gat:
            batch = batch.to(device)
            emb   = model_gat.encode(batch.x, batch.edge_index)[:batch.batch_size]
            emb_list_gat.append(emb.cpu())
    emb_gat = torch.cat(emb_list_gat, dim=0).numpy()
    torch.save(torch.tensor(emb_gat), GAT_EMB_CKPT)
    print(f'✓ GAT 임베딩 저장: {emb_gat.shape}')

EMB_GAT_COLS = [f'gat_{i}' for i in range(emb_gat.shape[1])]
df_emb_gat   = pd.DataFrame(emb_gat, columns=EMB_GAT_COLS)
df_emb_gat['patent_id'] = df_gnn['patent_id'].values

df_aug_gat = df_gnn[['patent_id', 'is_litigated'] + PATENT_FEAT_COLS].merge(
    df_emb_gat, on='patent_id', how='left')

FEAT_GAT  = PATENT_FEAT_COLS + EMB_GAT_COLS
aug_tr_gat = df_aug_gat[df_aug_gat['patent_id'].isin(train_pids)].copy()
aug_te_gat = df_aug_gat[df_aug_gat['patent_id'].isin(test_pids)].copy()
print(f'aug_train: {len(aug_tr_gat):,}  aug_test: {len(aug_te_gat):,}')

best_set    = max(m1_results, key=lambda k: m1_results[k]['test_auc'])
base_params = m1_results[best_set]['best_params'].copy()
base_params.update({'device': DEVICE, 'n_jobs': -1, 'random_state': RANDOM_STATE, 'verbose': -1})

lgb_gat = lgb.LGBMClassifier(**base_params)
lgb_gat.fit(aug_tr_gat[FEAT_GAT], aug_tr_gat['is_litigated'])

y_te_g    = aug_te_gat['is_litigated'].values
proba_gat = lgb_gat.predict_proba(aug_te_gat[FEAT_GAT])[:, 1]
auc_gat   = roc_auc_score(y_te_g, proba_gat)
ap_gat    = average_precision_score(y_te_g, proba_gat)

tr_proba_gat = lgb_gat.predict_proba(aug_tr_gat[FEAT_GAT])[:, 1]
best_thr_gat = threshold_metrics(aug_tr_gat['is_litigated'].values, tr_proba_gat)['threshold']
te_m_gat     = threshold_metrics(y_te_g, proba_gat, threshold=best_thr_gat)
te_m_gat_05  = threshold_metrics(y_te_g, proba_gat, threshold=0.5)

# ── 전체 비교표 ────────────────────────────────────────────────────────────
best_ml = m1_results[best_set]
thr_ml  = best_ml['threshold']
W = 100
print('\n' + '='*W)
print(f'{"모델":26s}  {"Thr":5s}  {"AUC":6s}  {"AP":6s}  {"Prec":6s}  {"Rec":6s}  {"F1":6s}  {"MCC":6s}  {"Gmean":6s}')
print('-'*W)
print(f'{"논문(기존)":26s}  {"0.50":5s}  {"":6s}  {"":6s}  {"0.016":6s}  {"0.629":6s}  {"0.031":6s}  {"0.073":6s}  {"0.697":6s}')
print('-'*W)
# LGB
print(f'{"LGB (22 feat, Gmean-opt)":26s}  {thr_ml:.2f}   '
      f'{best_ml["test_auc"]:.4f}  {best_ml["test_ap"]:.4f}  '
      f'{best_ml.get("test_Prec",float("nan")):.4f}  {best_ml.get("test_Rec",float("nan")):.4f}  '
      f'{best_ml.get("test_F1",float("nan")):.4f}  {best_ml.get("test_MCC",float("nan")):.4f}  '
      f'{best_ml.get("test_Gmean",float("nan")):.4f}')
print(f'{"LGB (22 feat, thr=0.50)":26s}  {"0.50":5s}  '
      f'{best_ml["test_auc"]:.4f}  {best_ml["test_ap"]:.4f}  '
      f'{best_ml.get("test05_Prec",float("nan")):.4f}  {best_ml.get("test05_Rec",float("nan")):.4f}  '
      f'{best_ml.get("test05_F1",float("nan")):.4f}  {best_ml.get("test05_MCC",float("nan")):.4f}  '
      f'{best_ml.get("test05_Gmean",float("nan")):.4f}')
# SAGE (있으면)
if 'auc_g' in dir():
    print(f'{"SAGE+LGB (Gmean-opt)":26s}  {best_thr_g:.2f}   '
          f'{auc_g:.4f}  {ap_g:.4f}  '
          f'{te_m_g["Prec"]:.4f}  {te_m_g["Rec"]:.4f}  '
          f'{te_m_g["F1"]:.4f}  {te_m_g["MCC"]:.4f}  {te_m_g["Gmean"]:.4f}')
    print(f'{"SAGE+LGB (thr=0.50)":26s}  {"0.50":5s}  '
          f'{auc_g:.4f}  {ap_g:.4f}  '
          f'{te_m_g_05["Prec"]:.4f}  {te_m_g_05["Rec"]:.4f}  '
          f'{te_m_g_05["F1"]:.4f}  {te_m_g_05["MCC"]:.4f}  {te_m_g_05["Gmean"]:.4f}')
# GAT
print(f'{"GAT+LGB (Gmean-opt)":26s}  {best_thr_gat:.2f}   '
      f'{auc_gat:.4f}  {ap_gat:.4f}  '
      f'{te_m_gat["Prec"]:.4f}  {te_m_gat["Rec"]:.4f}  '
      f'{te_m_gat["F1"]:.4f}  {te_m_gat["MCC"]:.4f}  {te_m_gat["Gmean"]:.4f}')
print(f'{"GAT+LGB (thr=0.50)":26s}  {"0.50":5s}  '
      f'{auc_gat:.4f}  {ap_gat:.4f}  '
      f'{te_m_gat_05["Prec"]:.4f}  {te_m_gat_05["Rec"]:.4f}  '
      f'{te_m_gat_05["F1"]:.4f}  {te_m_gat_05["MCC"]:.4f}  {te_m_gat_05["Gmean"]:.4f}')
print('='*W)


---
## Step 15: IPC 노드 추가
- IPC Subclass(예: G06F) 단위로 IPC 노드 생성
- 엣지: `patent --has_ipc--> ipc` / `ipc --rev_has_ipc--> patent`
- citation도 양방향으로 확장: `patent --cites--> patent` 
- 모델: GATFraudHetero (HeteroConv + GATConv)
- HeteroGAT: num_neighbors [5,3,2], batch_size:256, AMO (X) 

- (GATFraud: num_neighbors [10,10,5], batch_size:1024, AMP (O) )


In [ ]:
# [Cell 34]
# 현재 특허 데이터의 IPC Subclass 종류 파악
# IPC Subclass = section + ipc_class(2자리 zero-pad) + subclass  →  예: G06F, A61K
# 표준 IPC 섹션: A(생활필수품) B(처리조작) C(화학야금) D(섬유지류)
#               E(고정구조물) F(기계공학) G(물리학) H(전기)
import pandas as pd
from pathlib import Path

IPC_TSV  = Path('../src/g_ipc_at_issue.tsv')
IPC_CKPT = Path(DATA_DIR) / 'ipc_subclass_map.parquet'

if IPC_CKPT.exists():
    df_ipc = pd.read_parquet(IPC_CKPT)
    print(f'✓ IPC 맵 로드: {IPC_CKPT}')
else:
    our_pids = set(pd.read_parquet(f'{DATA_DIR}/patent_features.parquet',
                                   columns=['patent_id'])['patent_id'].tolist())
    chunks = []
    for chunk in pd.read_csv(IPC_TSV, sep='\t',
                             usecols=['patent_id', 'section', 'ipc_class', 'subclass'],
                             chunksize=500_000, on_bad_lines='skip', engine='python'):
        chunk['patent_id'] = pd.to_numeric(chunk['patent_id'], errors='coerce')
        chunk = chunk.dropna(subset=['patent_id', 'section', 'ipc_class', 'subclass'])
        chunk['patent_id'] = chunk['patent_id'].astype('int64')
        chunk = chunk[chunk['patent_id'].isin(our_pids)]
        chunks.append(chunk)
    raw = pd.concat(chunks, ignore_index=True)

    # G06F 형태로 합치기: section + ipc_class(2자리 zero-pad) + subclass
    raw['ipc_subclass'] = (raw['section'].str.strip()
                           + raw['ipc_class'].astype(str).str.strip().str.zfill(2)
                           + raw['subclass'].str.strip())

    df_ipc = raw[['patent_id', 'ipc_subclass']].drop_duplicates().reset_index(drop=True)
    df_ipc.to_parquet(IPC_CKPT, index=False)
    print(f'✓ IPC 맵 저장: {IPC_CKPT}')

# ── 통계 출력 ──────────────────────────────────────────────────────────────
n_patents_with_ipc = df_ipc['patent_id'].nunique()
n_subclass         = df_ipc['ipc_subclass'].nunique()
n_edges            = len(df_ipc)

print(f'\n=== IPC Subclass 현황 ===')
print(f'  IPC 있는 특허:   {n_patents_with_ipc:>10,} / 5,568,384')
print(f'  unique subclass: {n_subclass:>10,}  → IPC 노드 수')
print(f'  has_ipc 엣지:    {n_edges:>10,}  (rev_has_ipc까지 ×2 = {n_edges*2:,})')

# 표준 IPC 섹션(A-H)만 필터
df_std = df_ipc[df_ipc['ipc_subclass'].str[0].isin(list('ABCDEFGH'))]
print(f'\n표준 IPC(A-H) 필터 후:')
print(f'  unique subclass: {df_std["ipc_subclass"].nunique():>10,}')
print(f'  has_ipc 엣지:    {len(df_std):>10,}')

print(f'\nTop 15 subclass (특허 수 기준):')
top = df_std['ipc_subclass'].value_counts().head(15)
for sc, cnt in top.items():
    bar = '█' * int(cnt / top.max() * 30)
    print(f'  {sc:6s}  {cnt:8,}  {bar}')

print(f'\n섹션별 subclass 수 (표준 A-H):')
df_std2 = df_std.copy()
df_std2['section'] = df_std2['ipc_subclass'].str[0]
print(df_std2.groupby('section')['ipc_subclass'].nunique().sort_values(ascending=False))


In [ ]:
# [Cell 35]
# 공식 IPC subclass(639개) 기준으로 특허-IPC 매핑 구축 (ipc_sequence 포함)
import pandas as pd
from pathlib import Path

IPC_MAP_CKPT = Path(DATA_DIR) / 'ipc_subclass_map.parquet'

if IPC_MAP_CKPT.exists():
    df_ipc = pd.read_parquet(IPC_MAP_CKPT)
    print(f'✓ IPC 맵 로드: {IPC_MAP_CKPT}')
    print(f'컬럼: {df_ipc.columns.tolist()}')
else:
    valid_set = set(pd.read_parquet(f'{DATA_DIR}/valid_ipc_subclass.parquet')['ipc_subclass'].tolist())
    print(f'공식 IPC subclass: {len(valid_set)}개')

    US_IPC_TSV = Path('../src/g_ipc_at_issue.tsv')
    our_pids = set(pd.read_parquet(f'{DATA_DIR}/patent_features.parquet',
                                   columns=['patent_id'])['patent_id'].tolist())

    # TSV 컬럼 확인
    _head = pd.read_csv(US_IPC_TSV, sep='\t', nrows=0, engine='python')
    print(f'TSV 컬럼: {_head.columns.tolist()}')

    chunks = []
    total_rows = 0
    after_fmt = 0
    after_pid = 0
    after_valid = 0

    for chunk in pd.read_csv(US_IPC_TSV, sep='\t', on_bad_lines='skip',
                             engine='python', chunksize=500_000):
        total_rows += len(chunk)
        s   = chunk['section'].astype(str).str.strip().str.upper()
        ic  = chunk['ipc_class'].astype(str).str.strip()
        sc  = chunk['subclass'].astype(str).str.strip().str.upper()
        seq = pd.to_numeric(chunk['ipc_sequence'], errors='coerce')
        mask = (s.str.match(r'^[A-H]$') &
                ic.str.match(r'^\d{1,2}$') &
                sc.str.match(r'^[A-Z]$') &
                seq.notna())
        after_fmt += mask.sum()

        tmp = pd.DataFrame({
            'patent_id':    pd.to_numeric(chunk.loc[mask, 'patent_id'], errors='coerce').values,
            'ipc_subclass': (s[mask] + ic[mask].str.zfill(2) + sc[mask]).values,
            'ipc_sequence': seq[mask].values,
        })
        tmp = tmp.dropna(subset=['patent_id'])
        tmp['patent_id']    = tmp['patent_id'].astype('int64')
        tmp['ipc_sequence'] = tmp['ipc_sequence'].astype('int32')

        tmp_pid = tmp[tmp['patent_id'].isin(our_pids)]
        after_pid += len(tmp_pid)

        tmp_final = tmp_pid[tmp_pid['ipc_subclass'].isin(valid_set)]
        after_valid += len(tmp_final)
        chunks.append(tmp_final)

    df_ipc = pd.concat(chunks).drop_duplicates(subset=['patent_id', 'ipc_subclass']).reset_index(drop=True)
    df_ipc.to_parquet(IPC_MAP_CKPT, index=False)
    print(f'✓ IPC 맵 저장: {IPC_MAP_CKPT}')
    print(f'\n[필터링 통계]')
    print(f'  전체 TSV 행:         {total_rows:>12,}  (100.0%)')
    print(f'  형식 통과(fmt mask): {after_fmt:>12,}  ({after_fmt/total_rows*100:.1f}%)')
    print(f'  우리 특허 해당:      {after_pid:>12,}  ({after_pid/total_rows*100:.1f}%)')
    print(f'  공식 IPC 필터 후:    {after_valid:>12,}  ({after_valid/total_rows*100:.1f}%)')

print(f'\nIPC 있는 특허:   {df_ipc["patent_id"].nunique():,}')
print(f'unique subclass: {df_ipc["ipc_subclass"].nunique()}')
print(f'patent-ipc 엣지: {len(df_ipc):,}')
print(f'ipc_sequence 분포:\n{df_ipc["ipc_sequence"].value_counts().sort_index().head(10)}')
print(f'\nTop 10 subclass:')
print(df_ipc['ipc_subclass'].value_counts().head(10))

In [ ]:
# [Cell 36]
# IPC 노드 피처: BGE-Large 텍스트 임베딩 (BAAI/bge-large-en-v1.5, 1024-dim)
from pathlib import Path
import pandas as pd
import numpy as np

IPC_FEAT_CKPT = Path(DATA_DIR) / 'ipc_node_feat.parquet'

if IPC_FEAT_CKPT.exists():
    df_ipc_feat = pd.read_parquet(IPC_FEAT_CKPT)
    print(f'✓ IPC 피처 로드: shape={df_ipc_feat.shape}')
else:
    csv_path = Path(DATA_DIR) / 'gnn/ipc_subclass_mapping.csv'
    df_desc = pd.read_csv(csv_path, header=None, names=['ipc_subclass', 'description'])
    print(f'CSV 로드: {len(df_desc)}개')

    valid_subs = set(pd.read_parquet(f'{DATA_DIR}/ipc_subclass_map.parquet')['ipc_subclass'].unique())
    df_desc = df_desc[df_desc['ipc_subclass'].isin(valid_subs)].reset_index(drop=True)
    print(f'우리 데이터 매칭: {len(df_desc)}개 subclass')

    texts = df_desc['description'].tolist()

    try:
        from sentence_transformers import SentenceTransformer
        emb_model = SentenceTransformer('BAAI/bge-large-en-v1.5')
        embeddings = emb_model.encode(texts, batch_size=64, normalize_embeddings=True,
                                      show_progress_bar=True)
        embeddings = embeddings.astype('float32')
        print(f'BGE-Large 임베딩: {embeddings.shape}')
    except ImportError:
        print('sentence-transformers 없음 → pip install sentence-transformers')
        raise

    feat_cols = [f'f{i}' for i in range(embeddings.shape[1])]
    df_ipc_feat = pd.DataFrame(embeddings, columns=feat_cols)
    df_ipc_feat.insert(0, 'ipc_subclass', df_desc['ipc_subclass'].values)
    df_ipc_feat.to_parquet(IPC_FEAT_CKPT, index=False)
    print(f'✓ IPC 피처 저장: {IPC_FEAT_CKPT}')

IPC_FEAT_DIM = df_ipc_feat.shape[1] - 1
print(f'IPC 노드 수: {len(df_ipc_feat)}, 피처 차원: {IPC_FEAT_DIM}')

In [ ]:
# [Cell 37]
# HeteroData 구축: patent + IPC 노드, cites/cited_by/has_ipc/rev_has_ipc 엣지
import torch
import pandas as pd
from pathlib import Path
from torch_geometric.data import HeteroData

HETERO_CKPT = GNN_CKPT_DIR / 'hetero_data.pt'
META_CKPT   = GNN_CKPT_DIR / 'hetero_meta.pt'

if HETERO_CKPT.exists() and META_CKPT.exists():
    hetero_data = torch.load(HETERO_CKPT, weights_only=False)
    meta        = torch.load(META_CKPT, weights_only=False)
    pid_to_idx  = meta['pid_to_idx']
    ipc_to_idx  = meta['ipc_to_idx']
    print('✓ HeteroData 로드')
else:
    # ── Patent 노드 ───────────────────────────────────────────
    feat_df = pd.read_parquet(f'{DATA_DIR}/patent_features.parquet',
                              columns=['patent_id'] + PATENT_FEAT_COLS + ['is_litigated']).fillna(0)
    pid_to_idx = {int(p): i for i, p in enumerate(feat_df['patent_id'])}

    x_patent  = torch.tensor(feat_df[PATENT_FEAT_COLS].values, dtype=torch.float32)
    y_patent  = torch.tensor(feat_df['is_litigated'].values,   dtype=torch.long)

    train_pids = set(pd.read_parquet(f'{DATA_DIR}/split_train_pids.parquet')['patent_id'].tolist())
    test_pids  = set(pd.read_parquet(f'{DATA_DIR}/split_test.parquet')['patent_id'].tolist())
    train_mask = torch.tensor([int(p) in train_pids for p in feat_df['patent_id']], dtype=torch.bool)
    test_mask  = torch.tensor([int(p) in test_pids  for p in feat_df['patent_id']], dtype=torch.bool)
    print(f'patent 노드: {len(feat_df):,}  (train={train_mask.sum():,}, test={test_mask.sum():,})')

    # ── IPC 노드 ──────────────────────────────────────────────
    df_ipc_feat  = pd.read_parquet(f'{DATA_DIR}/ipc_node_feat.parquet')
    ipc_list     = df_ipc_feat['ipc_subclass'].tolist()
    ipc_to_idx   = {ipc: i for i, ipc in enumerate(ipc_list)}
    ipc_fcols    = [c for c in df_ipc_feat.columns if c != 'ipc_subclass']
    x_ipc        = torch.tensor(df_ipc_feat[ipc_fcols].values, dtype=torch.float32)
    print(f'ipc  노드: {len(ipc_list)}  (dim={x_ipc.shape[1]})')

    # ── Citation 엣지 ─────────────────────────────────────────
    ei_cite     = torch.load(GNN_CKPT_DIR / 'edge_index.pt')   # [2, E]
    print(f'cites 엣지: {ei_cite.shape[1]:,}')

    # ── IPC 엣지 (edge_attr = 1 / (seq+1)) ───────────────────
    df_ipc_map = pd.read_parquet(f'{DATA_DIR}/ipc_subclass_map.parquet')
    df_ipc_map = df_ipc_map[
        df_ipc_map['patent_id'].isin(pid_to_idx) &
        df_ipc_map['ipc_subclass'].isin(ipc_to_idx)
    ].copy()

    src_p = torch.tensor([pid_to_idx[int(p)] for p in df_ipc_map['patent_id']], dtype=torch.long)
    dst_i = torch.tensor([ipc_to_idx[s]      for s in df_ipc_map['ipc_subclass']], dtype=torch.long)
    w     = torch.tensor(1.0 / (df_ipc_map['ipc_sequence'].values + 1),
                         dtype=torch.float32).unsqueeze(1)  # [E, 1]
    print(f'has_ipc 엣지: {len(src_p):,}')

    # ── 조립 ──────────────────────────────────────────────────
    hetero_data = HeteroData()
    hetero_data['patent'].x          = x_patent
    hetero_data['patent'].y          = y_patent
    hetero_data['patent'].train_mask = train_mask
    hetero_data['patent'].test_mask  = test_mask
    hetero_data['ipc'].x             = x_ipc

    hetero_data['patent', 'cites',       'patent'].edge_index = ei_cite
    hetero_data['patent', 'has_ipc',     'ipc'   ].edge_index = torch.stack([src_p, dst_i])
    hetero_data['patent', 'has_ipc',     'ipc'   ].edge_attr  = w
    hetero_data['ipc',    'rev_has_ipc', 'patent'].edge_index = torch.stack([dst_i, src_p])
    hetero_data['ipc',    'rev_has_ipc', 'patent'].edge_attr  = w

    torch.save(hetero_data, HETERO_CKPT)
    torch.save({'pid_to_idx': pid_to_idx, 'ipc_to_idx': ipc_to_idx}, META_CKPT)
    print('✓ HeteroData 저장')

print(hetero_data)

In [ ]:
# [Cell 38]
# GATFraudHetero 모델 (HeteroConv + GATConv, 3-layer, BN, Focal Loss)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GATConv
from torch_geometric.loader import NeighborLoader

PATENT_IN = len(PATENT_FEAT_COLS)
IPC_IN    = hetero_data['ipc'].x.shape[1]
HIDDEN_H  = 256
HEADS_H   = 4

class GATFraudHetero(torch.nn.Module):
    def __init__(self, patent_in=PATENT_IN, ipc_in=IPC_IN,
                 hidden=HIDDEN_H, heads=HEADS_H, drop=0.3):
        super().__init__()
        self.drop        = drop
        hh               = hidden // heads
        self.patent_proj = nn.Linear(patent_in, hidden)
        self.ipc_proj    = nn.Linear(ipc_in,    hidden)

        def _hconv():
            return HeteroConv({
                ('patent', 'cites',       'patent'): GATConv(hidden, hh, heads=heads, add_self_loops=False),
                ('patent', 'has_ipc',     'ipc'):    GATConv(hidden, hh, heads=heads, add_self_loops=False, edge_dim=1),
                ('ipc',    'rev_has_ipc', 'patent'): GATConv(hidden, hh, heads=heads, add_self_loops=False, edge_dim=1),
            }, aggr='sum')

        self.conv1 = _hconv()
        self.conv2 = _hconv()
        self.conv3 = _hconv()

        self.bn = nn.ModuleDict({
            'l1_patent': nn.BatchNorm1d(hidden), 'l1_ipc': nn.BatchNorm1d(hidden),
            'l2_patent': nn.BatchNorm1d(hidden), 'l2_ipc': nn.BatchNorm1d(hidden),
            'l3_patent': nn.BatchNorm1d(hidden),
        })
        self.clf = nn.Linear(hidden, 1)

    def _bn_act(self, x_dict, prefix, residual=None):
        out = {}
        for k, x in x_dict.items():
            key = f'{prefix}_{k}'
            if key in self.bn:
                x = F.elu(self.bn[key](x))
                if residual and k in residual:
                    x = x + residual[k]
                x = F.dropout(x, p=self.drop, training=self.training)
            out[k] = x
        return out

    def encode(self, x_dict, edge_index_dict, edge_attr_dict=None):
        x = {
            'patent': F.dropout(F.elu(self.patent_proj(x_dict['patent'])), p=self.drop, training=self.training),
            'ipc':    F.dropout(F.elu(self.ipc_proj(x_dict['ipc'])),       p=self.drop, training=self.training),
        }
        x1 = self._bn_act(self.conv1(x,  edge_index_dict, edge_attr_dict), 'l1')
        x2 = self._bn_act(self.conv2(x1, edge_index_dict, edge_attr_dict), 'l2', residual=x1)
        x3 = self.conv3(x2, edge_index_dict, edge_attr_dict)
        return F.elu(self.bn['l3_patent'](x3['patent']))

    def forward(self, x_dict, edge_index_dict, edge_attr_dict=None):
        return self.clf(self.encode(x_dict, edge_index_dict, edge_attr_dict)).squeeze(-1)

model_hetero = GATFraudHetero().to(device)
print(f'파라미터 수: {sum(p.numel() for p in model_hetero.parameters() if p.requires_grad):,}')

# NeighborLoader
_nn_h      = {et: [10, 10, 5] for et in hetero_data.edge_types}
train_idx_h = hetero_data['patent'].train_mask.nonzero(as_tuple=True)[0]
test_idx_h  = hetero_data['patent'].test_mask.nonzero(as_tuple=True)[0]

train_loader_h = NeighborLoader(hetero_data, num_neighbors=_nn_h,
                                 batch_size=1024, input_nodes=('patent', train_idx_h), shuffle=True)
test_loader_h  = NeighborLoader(hetero_data, num_neighbors=_nn_h,
                                 batch_size=2048, input_nodes=('patent', test_idx_h),  shuffle=False)
print(f'train batches: {len(train_loader_h)}, test batches: {len(test_loader_h)}')

In [ ]:
# [Cell 39]
# GATFraudHetero 학습 (fp32, num_workers=8) + 임베딩 추출
import torch, os
from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.loader import NeighborLoader

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

HETERO_MDL_CKPT = GNN_CKPT_DIR / 'model_hetero_gat.pt'
HETERO_EMB_CKPT = GNN_CKPT_DIR / 'embeddings_hetero_gat.pt'

_nn_h       = {et: [5, 3, 2] for et in hetero_data.edge_types}
train_idx_h = hetero_data['patent'].train_mask.nonzero(as_tuple=True)[0]
test_idx_h  = hetero_data['patent'].test_mask.nonzero(as_tuple=True)[0]

train_loader_h = NeighborLoader(hetero_data, num_neighbors=_nn_h,
                                 batch_size=256, input_nodes=('patent', train_idx_h),
                                 shuffle=True, num_workers=8)
test_loader_h  = NeighborLoader(hetero_data, num_neighbors=_nn_h,
                                 batch_size=512, input_nodes=('patent', test_idx_h),
                                 shuffle=False, num_workers=8)
print(f'train batches: {len(train_loader_h)}, test batches: {len(test_loader_h)}')

_y_tr   = hetero_data['patent'].y[hetero_data['patent'].train_mask]
pos_w_h = float(((_y_tr == 0).sum()) / (_y_tr.sum()))
print(f'pos_weight={pos_w_h:.1f}')

def _ea(batch):
    return {et: getattr(batch[et], 'edge_attr', None) for et in batch.edge_types}

def train_epoch_h(model, loader, opt):
    model.train()
    total = 0
    for batch in loader:
        batch = batch.to(device)
        n     = batch['patent'].batch_size
        opt.zero_grad()
        logits = model(batch.x_dict, batch.edge_index_dict, _ea(batch))[:n]
        labels = batch['patent'].y[:n].float()
        loss   = focal_loss(logits, labels, gamma=2.0, pos_weight=pos_w_h)
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_h(model, loader):
    model.eval()
    probs, ys = [], []
    for batch in loader:
        batch = batch.to(device)
        n     = batch['patent'].batch_size
        logits = model(batch.x_dict, batch.edge_index_dict, _ea(batch))[:n]
        probs.append(torch.sigmoid(logits).cpu())
        ys.append(batch['patent'].y[:n].cpu())
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

if HETERO_MDL_CKPT.exists():
    model_hetero.load_state_dict(torch.load(HETERO_MDL_CKPT, map_location=device, weights_only=True))
    print('✓ 체크포인트 로드')
else:
    opt_h = torch.optim.Adam(model_hetero.parameters(), lr=1e-3, weight_decay=1e-4)
    sch_h = torch.optim.lr_scheduler.CosineAnnealingLR(opt_h, T_max=30)
    best_auc_h = 0
    for ep in range(1, 31):
        loss = train_epoch_h(model_hetero, train_loader_h, opt_h)
        sch_h.step()
        if ep % 5 == 0:
            proba, y = eval_h(model_hetero, test_loader_h)
            auc = roc_auc_score(y, proba)
            ap  = average_precision_score(y, proba)
            print(f'Epoch {ep:02d} | loss={loss:.4f} | AUC={auc:.4f} | AP={ap:.4f}')
            if auc > best_auc_h:
                best_auc_h = auc
                torch.save(model_hetero.state_dict(), HETERO_MDL_CKPT)
    print(f'✓ 학습 완료 (best AUC={best_auc_h:.4f})')
    model_hetero.load_state_dict(torch.load(HETERO_MDL_CKPT, map_location=device, weights_only=True))

# 임베딩 추출
if HETERO_EMB_CKPT.exists():
    emb_hetero_gat = torch.load(HETERO_EMB_CKPT, weights_only=True)
    print(f'✓ 임베딩 로드: {emb_hetero_gat.shape}')
else:
    N_all        = hetero_data['patent'].num_nodes
    all_loader_h = NeighborLoader(hetero_data, num_neighbors=_nn_h,
                                   batch_size=1024, input_nodes=('patent', torch.arange(N_all)),
                                   shuffle=False, num_workers=8)
    model_hetero.eval()
    emb_hetero_gat = torch.zeros(N_all, HIDDEN_H)
    with torch.no_grad():
        for batch in all_loader_h:
            batch = batch.to(device)
            n     = batch['patent'].batch_size
            emb   = model_hetero.encode(batch.x_dict, batch.edge_index_dict, _ea(batch))[:n]
            emb_hetero_gat[batch['patent'].n_id[:n]] = emb.cpu()
    torch.save(emb_hetero_gat, HETERO_EMB_CKPT)
    print(f'✓ 임베딩 저장: {emb_hetero_gat.shape}')

HeteroGNN 직접 예측

In [18]:
# [Cell 40]
# HeteroGAT 평가 (threshold=0.5)
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_score, recall_score,
                              matthews_corrcoef, confusion_matrix)
import numpy as np

proba_h, y_h = eval_h(model_hetero, test_loader_h)
pred_h = (proba_h >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_h, pred_h).ravel()
recall_h      = tp / (tp + fn)
specificity_h = tn / (tn + fp)
gmean_h       = np.sqrt(recall_h * specificity_h)

print(f'AUC:         {roc_auc_score(y_h, proba_h):.4f}')
print(f'AP:          {average_precision_score(y_h, proba_h):.4f}')
print(f'Precision:   {precision_score(y_h, pred_h):.4f}')
print(f'Recall:      {recall_h:.4f}')
print(f'Specificity: {specificity_h:.4f}')
print(f'G-mean:      {gmean_h:.4f}')
print(f'F1:          {f1_score(y_h, pred_h):.4f}')
print(f'MCC:         {matthews_corrcoef(y_h, pred_h):.4f}')

AUC:         0.7405
AP:          0.0181
Precision:   0.0113
Recall:      0.7549
Specificity: 0.5982
G-mean:      0.6720
F1:          0.0222
MCC:         0.0557


HeteroGAT+LightGBM

In [ ]:
# [Cell 41]
# HeteroGAT 임베딩 + 원본 feature → LightGBM
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_score, recall_score,
                              matthews_corrcoef, confusion_matrix)

feat_df_h = pd.read_parquet(f'{DATA_DIR}/patent_features.parquet',
                              columns=['patent_id'] + PATENT_FEAT_COLS).fillna(0)

train_mask_h = hetero_data['patent'].train_mask.numpy()
test_mask_h  = hetero_data['patent'].test_mask.numpy()
y_all_h      = hetero_data['patent'].y.numpy()

feat_np_h = feat_df_h[PATENT_FEAT_COLS].values
emb_np_h  = emb_hetero_gat.numpy()

X_tr_h = np.hstack([feat_np_h[train_mask_h], emb_np_h[train_mask_h]])
X_te_h = np.hstack([feat_np_h[test_mask_h],  emb_np_h[test_mask_h]])
y_tr_h = y_all_h[train_mask_h]
y_te_h = y_all_h[test_mask_h]

# best_params_m1 있으면 사용, 없으면 기본값
try:
    lgb_p = {**best_params_m1}
except NameError:
    lgb_p = {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
              'num_leaves': 63, 'learning_rate': 0.05, 'n_estimators': 500}

lgb_p['scale_pos_weight'] = float((y_tr_h == 0).sum() / y_tr_h.sum())
lgb_p['verbosity'] = -1

lgb_h = lgb.train(lgb_p,
                   lgb.Dataset(X_tr_h, y_tr_h),
                   num_boost_round=500,
                   valid_sets=[lgb.Dataset(X_te_h, y_te_h)],
                   callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(period=-1)])

proba_lgb_h = lgb_h.predict(X_te_h)
pred_lgb_h  = (proba_lgb_h >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_te_h, pred_lgb_h).ravel()
recall_v      = tp / (tp + fn)
specificity_v = tn / (tn + fp)
gmean_v       = np.sqrt(recall_v * specificity_v)

print(f'HeteroGAT + LGB (threshold=0.5)')
print(f'AUC:         {roc_auc_score(y_te_h, proba_lgb_h):.4f}')
print(f'AP:          {average_precision_score(y_te_h, proba_lgb_h):.4f}')
print(f'Precision:   {precision_score(y_te_h, pred_lgb_h):.4f}')
print(f'Recall:      {recall_v:.4f}')
print(f'Specificity: {specificity_v:.4f}')
print(f'G-mean:      {gmean_v:.4f}')
print(f'F1:          {f1_score(y_te_h, pred_lgb_h):.4f}')
print(f'MCC:         {matthews_corrcoef(y_te_h, pred_lgb_h):.4f}')

In [ ]:
# [Cell 42]
# HeteroSAGE: 기존 GraphSAGE(Cell 30)와 동일 구조 + IPC 노드 추가
# hidden=128, emb=64, batch=512, epochs=50, lr=1e-3, BCE+pos_weight, neighbors [25,10]
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.loader import NeighborLoader
from sklearn.metrics import roc_auc_score, average_precision_score

HSAGE_MDL_CKPT = GNN_CKPT_DIR / 'model_hetero_sage.pt'
HSAGE_EMB_CKPT = GNN_CKPT_DIR / 'embeddings_hetero_sage.pt'
PATENT_IN = len(PATENT_FEAT_COLS)  # 22
EMB_DIM   = 64

class HeteroSAGE(torch.nn.Module):
    def __init__(self, patent_in=PATENT_IN, ipc_in=IPC_FEAT_DIM, hidden=128, emb=64, drop=0.3):
        super().__init__()
        self.drop        = drop
        # 두 노드 타입 차원이 달라서 hidden으로 먼저 통일
        self.patent_proj = nn.Linear(patent_in, hidden)
        self.ipc_proj    = nn.Linear(ipc_in,    hidden)

        self.conv1 = HeteroConv({
            ('patent', 'cites',       'patent'): SAGEConv(hidden, hidden),
            ('patent', 'has_ipc',     'ipc'):    SAGEConv(hidden, hidden),
            ('ipc',    'rev_has_ipc', 'patent'): SAGEConv(hidden, hidden),
        }, aggr='sum')
        self.conv2 = HeteroConv({
            ('patent', 'cites',       'patent'): SAGEConv(hidden, emb),
            ('patent', 'has_ipc',     'ipc'):    SAGEConv(hidden, emb),
            ('ipc',    'rev_has_ipc', 'patent'): SAGEConv(hidden, emb),
        }, aggr='sum')
        self.clf = nn.Linear(emb, 1)

    def encode(self, x_dict, edge_index_dict):
        x = {
            'patent': self.patent_proj(x_dict['patent']).relu(),
            'ipc':    self.ipc_proj(x_dict['ipc']).relu(),
        }
        x1 = {k: v.relu() for k, v in self.conv1(x,  edge_index_dict).items()}
        x1 = {k: F.dropout(v, p=self.drop, training=self.training) for k, v in x1.items()}
        x2 = self.conv2(x1, edge_index_dict)
        return x2['patent']

    def forward(self, x_dict, edge_index_dict):
        return self.clf(self.encode(x_dict, edge_index_dict)).squeeze(-1)

model_hsage = HeteroSAGE().to(device)
print(f'파라미터 수: {sum(p.numel() for p in model_hsage.parameters() if p.requires_grad):,}')

_nn_hs       = {et: [25, 10] for et in hetero_data.edge_types}
train_idx_hs = hetero_data['patent'].train_mask.nonzero(as_tuple=True)[0]
test_idx_hs  = hetero_data['patent'].test_mask.nonzero(as_tuple=True)[0]

train_loader_hs = NeighborLoader(hetero_data, num_neighbors=_nn_hs,
                                  batch_size=512, input_nodes=('patent', train_idx_hs),
                                  shuffle=True, num_workers=8)
test_loader_hs  = NeighborLoader(hetero_data, num_neighbors=_nn_hs,
                                  batch_size=2048, input_nodes=('patent', test_idx_hs),
                                  shuffle=False, num_workers=8)
print(f'train batches: {len(train_loader_hs)}, test batches: {len(test_loader_hs)}')

_y_tr_hs = hetero_data['patent'].y[hetero_data['patent'].train_mask].float()
pos_w_hs  = float(((_y_tr_hs == 0).sum()) / (_y_tr_hs.sum()))
print(f'pos_weight={pos_w_hs:.1f}')

def train_epoch_hs(model, loader, opt):
    model.train()
    total, n_batch = 0.0, 0
    for batch in loader:
        batch  = batch.to(device)
        n      = batch['patent'].batch_size
        opt.zero_grad()
        logits = model(batch.x_dict, batch.edge_index_dict)[:n]
        labels = batch['patent'].y[:n].float()
        loss   = F.binary_cross_entropy_with_logits(
            logits, labels,
            pos_weight=torch.tensor([pos_w_hs], device=device))
        loss.backward()
        opt.step()
        total += loss.item(); n_batch += 1
    return total / n_batch

@torch.no_grad()
def eval_hs(model, loader):
    model.eval()
    probs, ys = [], []
    for batch in loader:
        batch = batch.to(device)
        n     = batch['patent'].batch_size
        logits = model(batch.x_dict, batch.edge_index_dict)[:n]
        probs.append(torch.sigmoid(logits).cpu())
        ys.append(batch['patent'].y[:n].cpu())
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

if HSAGE_MDL_CKPT.exists():
    model_hsage.load_state_dict(torch.load(HSAGE_MDL_CKPT, map_location=device, weights_only=True))
    print('✓ 체크포인트 로드')
else:
    opt_hs  = torch.optim.Adam(model_hsage.parameters(), lr=1e-3, weight_decay=1e-5)
    EPOCHS  = 50
    best_auc_hs = 0
    for ep in range(1, EPOCHS + 1):
        loss = train_epoch_hs(model_hsage, train_loader_hs, opt_hs)
        print(f'  Epoch {ep:3d}/{EPOCHS}  loss={loss:.4f}', flush=True)
        if ep % 10 == 0:
            proba, y = eval_hs(model_hsage, test_loader_hs)
            auc = roc_auc_score(y, proba)
            ap  = average_precision_score(y, proba)
            print(f'  → AUC={auc:.4f} | AP={ap:.4f}')
            if auc > best_auc_hs:
                best_auc_hs = auc
                torch.save(model_hsage.state_dict(), HSAGE_MDL_CKPT)
    print(f'✓ 학습 완료 (best AUC={best_auc_hs:.4f})')
    model_hsage.load_state_dict(torch.load(HSAGE_MDL_CKPT, map_location=device, weights_only=True))

# 임베딩 추출 (64-dim)
if HSAGE_EMB_CKPT.exists():
    emb_hsage = torch.load(HSAGE_EMB_CKPT, weights_only=True)
    print(f'✓ 임베딩 로드: {emb_hsage.shape}')
else:
    N_all = hetero_data['patent'].num_nodes
    all_loader_hs = NeighborLoader(hetero_data, num_neighbors=_nn_hs,
                                    batch_size=4096, input_nodes=('patent', torch.arange(N_all)),
                                    shuffle=False, num_workers=8)
    model_hsage.eval()
    emb_hsage = torch.zeros(N_all, EMB_DIM)
    with torch.no_grad():
        for batch in all_loader_hs:
            batch = batch.to(device)
            n     = batch['patent'].batch_size
            emb   = model_hsage.encode(batch.x_dict, batch.edge_index_dict)[:n]
            emb_hsage[batch['patent'].n_id[:n]] = emb.cpu()
    torch.save(emb_hsage, HSAGE_EMB_CKPT)
    print(f'✓ 임베딩 저장: {emb_hsage.shape}')

In [ ]:
# [Cell 43]
# HeteroSAGE 임베딩 + 원본 feature → LightGBM
import lightgbm as lgb
import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_score, recall_score,
                              matthews_corrcoef, confusion_matrix)
import pandas as pd

feat_df_hs = pd.read_parquet(f'{DATA_DIR}/patent_features.parquet',
                               columns=['patent_id'] + PATENT_FEAT_COLS).fillna(0)

train_mask_hs = hetero_data['patent'].train_mask.numpy()
test_mask_hs  = hetero_data['patent'].test_mask.numpy()
y_all_hs      = hetero_data['patent'].y.numpy()

feat_np_hs = feat_df_hs[PATENT_FEAT_COLS].values
emb_np_hs  = emb_hsage.numpy()

X_tr_hs = np.hstack([feat_np_hs[train_mask_hs], emb_np_hs[train_mask_hs]])
X_te_hs = np.hstack([feat_np_hs[test_mask_hs],  emb_np_hs[test_mask_hs]])
y_tr_hs = y_all_hs[train_mask_hs]
y_te_hs = y_all_hs[test_mask_hs]

try:
    lgb_p_hs = {**best_params_m1}
except NameError:
    lgb_p_hs = {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
                 'num_leaves': 63, 'learning_rate': 0.05}

lgb_p_hs['scale_pos_weight'] = float((y_tr_hs == 0).sum() / y_tr_hs.sum())
lgb_p_hs['verbosity'] = -1

lgb_hs = lgb.train(lgb_p_hs,
                    lgb.Dataset(X_tr_hs, y_tr_hs),
                    num_boost_round=500,
                    valid_sets=[lgb.Dataset(X_te_hs, y_te_hs)],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                                lgb.log_evaluation(period=-1)])

proba_hs = lgb_hs.predict(X_te_hs)
pred_hs  = (proba_hs >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_te_hs, pred_hs).ravel()
recall_hs      = tp / (tp + fn)
specificity_hs = tn / (tn + fp)
gmean_hs       = np.sqrt(recall_hs * specificity_hs)

print(f'HeteroSAGE + LGB (threshold=0.5)')
print(f'AUC:         {roc_auc_score(y_te_hs, proba_hs):.4f}')
print(f'AP:          {average_precision_score(y_te_hs, proba_hs):.4f}')
print(f'Precision:   {precision_score(y_te_hs, pred_hs):.4f}')
print(f'Recall:      {recall_hs:.4f}')
print(f'Specificity: {specificity_hs:.4f}')
print(f'G-mean:      {gmean_hs:.4f}')
print(f'F1:          {f1_score(y_te_hs, pred_hs):.4f}')
print(f'MCC:         {matthews_corrcoef(y_te_hs, pred_hs):.4f}')